In [3]:
# =======================
# 🔹 Cell 1 - Upload & Inspect Zeek DNS Data
# =======================

from google.colab import files
import pandas as pd
import zipfile, io, os

# --- Step 1: Upload ZIP file ---
print("📁 لطفاً فایل ZIP دیتای خود را آپلود کنید:")
uploaded = files.upload()

# --- Step 2: Extract ZIP contents ---
for fname in uploaded.keys():
    zip_path = fname
    break  # only the first uploaded file

extract_path = "/content/zeek_data"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print(f"✅ فایل از حالت فشرده خارج شد در: {extract_path}")

# --- Step 3: Find the first CSV file ---
csv_files = [f for f in os.listdir(extract_path) if f.endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("❌ هیچ فایل CSV در فایل ZIP یافت نشد.")

csv_path = os.path.join(extract_path, csv_files[0])
print(f"🗂 اولین فایل CSV پیدا شده: {csv_path}")

# --- Step 4: Load and preview ---
df = pd.read_csv(csv_path)
print("\n🔹 ۵ سطر اول داده:")
display(df.head(5))

print("\n🔹 ۵ سطر آخر داده:")
display(df.tail(5))


📁 لطفاً فایل ZIP دیتای خود را آپلود کنید:


Saving merged (3)dns.zip to merged (3)dns.zip
✅ فایل از حالت فشرده خارج شد در: /content/zeek_data
🗂 اولین فایل CSV پیدا شده: /content/zeek_data/merged (3).csv

🔹 ۵ سطر اول داده:


,@timestamp,source.ip,zeek.dns.query,zeek.dns.rcode_name,zeek.dns.qtype_name,dns.question.subdomain,zeek.dns.rtt,zeek.dns.TTLs,zeek.dns.answers,network.transport,zeek.dns.rejected,zeek.dns.TC,destination.ip,destination.port
0,"Sep 22, 2025 @ 15:29:59.592",192.168.100.119,es.itp.local,NOERROR,AAAA,-,-,-,-,udp,False,False,192.168.100.5,53
1,"Sep 22, 2025 @ 15:29:59.592",192.168.100.119,es.itp.local,NOERROR,A,-,0.001,"3,600",192.168.100.112,udp,False,False,192.168.100.5,53
2,"Sep 22, 2025 @ 15:29:59.592",192.168.100.119,es.itp.local,NOERROR,AAAA,-,-,-,-,udp,False,False,192.168.100.5,53
3,"Sep 22, 2025 @ 15:29:59.592",192.168.100.119,es.itp.local,NOERROR,A,-,0.001,"3,600",192.168.100.112,udp,False,False,192.168.100.5,53
4,"Sep 22, 2025 @ 15:29:59.203",192.168.100.20,xmsecu.com,-,A,-,-,-,-,udp,False,False,217.218.127.127,53



🔹 ۵ سطر آخر داده:


,@timestamp,source.ip,zeek.dns.query,zeek.dns.rcode_name,zeek.dns.qtype_name,dns.question.subdomain,zeek.dns.rtt,zeek.dns.TTLs,zeek.dns.answers,network.transport,zeek.dns.rejected,zeek.dns.TC,destination.ip,destination.port
2236315,"Oct 20, 2025 @ 12:00:00.875",192.168.100.5,asia.pool.ntp.org,REFUSED,A,asia.pool,-,-,-,udp,False,False,77.104.104.104,53
2236316,"Oct 20, 2025 @ 12:00:00.874",192.168.100.9,asia.pool.ntp.org,NOERROR,AAAA,asia.pool,-,-,-,udp,False,False,8.8.8.8,53
2236317,"Oct 20, 2025 @ 12:00:00.874",192.168.100.9,asia.pool.ntp.org,NOERROR,AAAA,asia.pool,-,-,-,udp,False,False,8.8.8.8,53
2236318,"Oct 20, 2025 @ 12:00:00.874",192.168.100.9,asia.pool.ntp.org,-,A,asia.pool,-,-,-,udp,False,False,8.8.8.8,53
2236319,"Oct 20, 2025 @ 12:00:00.874",192.168.100.9,asia.pool.ntp.org,-,A,asia.pool,-,-,-,udp,False,False,8.8.8.8,53


In [2]:
# =======================
# 🔹 Cell 2 - Feature Engineering for Zeek.DNS Anomaly Detection
# =======================

import pandas as pd
import numpy as np
from collections import Counter
import re

# --- Load the extracted CSV from previous cell ---
csv_path = csv_path  # مسیر فایل از سلول قبل
df = pd.read_csv(csv_path)

print(f"✅ داده بارگذاری شد: {len(df)} رکورد\n")

# --- Clean up missing and placeholder values ---
df.replace("-", np.nan, inplace=True)

# حذف رکوردهای بدون IP یا query
df = df.dropna(subset=["source.ip", "zeek.dns.query"])
print(f"📊 بعد از پاکسازی: {len(df)} رکورد باقی ماند.\n")

# --- Helper functions ---
def entropy(s):
    """Shannon entropy of a string."""
    if not isinstance(s, str) or len(s) == 0:
        return np.nan
    p = np.array(list(Counter(s).values())) / len(s)
    return -(p * np.log2(p)).sum()

def ttl_mean(ttl_str):
    """Compute mean TTL from comma-separated list."""
    if not isinstance(ttl_str, str):
        return np.nan
    nums = [float(x) for x in re.findall(r"\d+", ttl_str)]
    return np.mean(nums) if len(nums) > 0 else np.nan

def ttl_std(ttl_str):
    """Compute std TTL from comma-separated list."""
    if not isinstance(ttl_str, str):
        return np.nan
    nums = [float(x) for x in re.findall(r"\d+", ttl_str)]
    return np.std(nums) if len(nums) > 0 else np.nan

def answer_count(ans):
    """Number of answers if list-like string."""
    if not isinstance(ans, str) or ans.strip() == "":
        return 0
    return len(ans.split(","))

# --- Create basic columns ---
df["domain_length"] = df["zeek.dns.query"].astype(str).str.len()
df["subdomain_depth"] = df["zeek.dns.query"].astype(str).apply(lambda x: x.count("."))
df["entropy"] = df["zeek.dns.query"].astype(str).apply(entropy)
df["ttl_mean"] = df["zeek.dns.TTLs"].apply(ttl_mean)
df["ttl_std"] = df["zeek.dns.TTLs"].apply(ttl_std)
df["answer_count"] = df["zeek.dns.answers"].apply(answer_count)

# --- Convert boolean columns to numeric ---
for col in ["zeek.dns.rejected", "zeek.dns.TC"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().map({"true": 1, "false": 0})
        df[col] = df[col].fillna(0)

# --- RTT numeric conversion ---
df["zeek.dns.rtt"] = pd.to_numeric(df["zeek.dns.rtt"], errors="coerce")

# --- Group by source.ip ---
agg_df = df.groupby("source.ip").agg(
    dns_count=("zeek.dns.query", "count"),
    unique_domains=("zeek.dns.query", pd.Series.nunique),
    nxdomain_ratio=("zeek.dns.rcode_name", lambda x: (x=="NXDOMAIN").mean()),
    refused_ratio=("zeek.dns.rcode_name", lambda x: (x=="REFUSED").mean()),
    avg_domain_len=("domain_length", "mean"),
    avg_entropy=("entropy", "mean"),
    avg_subdomain_depth=("subdomain_depth", "mean"),
    avg_rtt=("zeek.dns.rtt", "mean"),
    std_rtt=("zeek.dns.rtt", "std"),
    avg_ttl=("ttl_mean", "mean"),
    std_ttl=("ttl_std", "mean"),
    avg_answer_count=("answer_count", "mean"),
    tcp_ratio=("network.transport", lambda x: (x=="tcp").mean()),
    rejected_ratio=("zeek.dns.rejected", "mean"),
    tc_ratio=("zeek.dns.TC", "mean")
).reset_index()

print(f"✅ فیچرها ساخته شدند: {agg_df.shape[0]} IP و {agg_df.shape[1]-1} فیچر عددی\n")

# --- Preview first and last 5 rows ---
print("🔹 پنج سطر اول داده فیچر شده:")
display(agg_df.head(5))

print("\n🔹 پنج سطر آخر داده فیچر شده:")
display(agg_df.tail(5))


✅ داده بارگذاری شد: 2236320 رکورد

📊 بعد از پاکسازی: 2233400 رکورد باقی ماند.

✅ فیچرها ساخته شدند: 120 IP و 15 فیچر عددی

🔹 پنج سطر اول داده فیچر شده:


,source.ip,dns_count,unique_domains,nxdomain_ratio,refused_ratio,avg_domain_len,avg_entropy,avg_subdomain_depth,avg_rtt,std_rtt,avg_ttl,std_ttl,avg_answer_count,tcp_ratio,rejected_ratio,tc_ratio
0,10.10.10.18,22197,293,0.002883,0.0,29.430103,3.686298,2.836960,0.557186,1.347328,160.046091,89.028848,3.426049,0.000315,0.000000,0.00018
1,10.10.10.188,39235,540,0.003517,0.0,18.998037,3.531254,2.161845,0.091257,0.227075,281.107116,221.790480,0.589525,0.000000,0.056021,0.00000
2,10.10.10.189,2994,122,0.010020,0.0,22.714095,3.434668,2.491650,0.537155,1.731309,226.645156,139.814362,4.151637,0.000000,0.026720,0.00000
3,10.10.10.200,75504,431,0.008397,0.0,24.120113,3.721843,2.079850,1.440527,2.664045,162.794365,123.963758,4.480266,0.000000,0.013059,0.00000
4,10.10.10.21,1785,91,0.002241,0.0,21.278992,3.120689,1.539496,0.329068,0.885851,162.773505,70.287955,4.854902,0.000000,0.000000,0.00000



🔹 پنج سطر آخر داده فیچر شده:


,source.ip,dns_count,unique_domains,nxdomain_ratio,refused_ratio,avg_domain_len,avg_entropy,avg_subdomain_depth,avg_rtt,std_rtt,avg_ttl,std_ttl,avg_answer_count,tcp_ratio,rejected_ratio,tc_ratio
115,fe80::dfb5:36b7:1357:4152,10016,38,0.0,0.0,32.272165,3.871653,2.744609,0.508825,1.014215,122.491919,78.014769,0.944089,0.0,0.0,0.0
116,fe80::ed:dcc1:ee65:e5df,15370,11,0.0,0.0,23.987053,3.746225,1.883409,1.700952,2.961253,154.871525,118.756900,5.110085,0.0,0.0,0.0
117,fe80::ee64:5c7d:4fa5:3b7f,3231,86,0.0,0.0,32.957289,3.794544,1.513463,0.460896,0.432934,104.082883,25.879902,0.786753,0.0,0.0,0.0
118,fe80::f0fc:e4ff:fe2f:475b,224,2,0.0,0.0,40.000000,3.845620,2.000000,1.947286,0.953477,192.000000,194.596085,10.000000,0.0,0.0,0.0
119,fe80::f8e3:4ff:fe17:5e88,11655,10,0.0,0.0,36.131188,3.799200,3.496353,2.015875,2.173516,138.353594,124.805336,4.062462,0.0,0.0,0.0


In [3]:
# =======================
# 🔹 Cell 3 — Fixed & Optimized IF (fast TS, 5m window, bagged IFs) + better training
#     - Pre-filter common benign domains (reduce FP)
#     - Add DGA-focused features
#     - Slightly higher contamination to reduce FN
# =======================
import os, time, re
import numpy as np
import pandas as pd
from collections import Counter
from numpy.random import default_rng
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

t0 = time.time()

# ---------- 0) Expect raw dataframe `df` from previous cells ----------
df_raw = df.copy()

# ---------- 1) Minimal cleaning ----------
for col in ["zeek.dns.TTLs", "zeek.dns.answers", "zeek.dns.rtt", "zeek.dns.rejected", "zeek.dns.TC"]:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].replace("-", np.nan)

# ---------- 2) SUPER-FAST timestamp parsing (vectorized, 2-pass) ----------
ts_str = df_raw["@timestamp"].astype(str).str.replace(" @ ", " ", regex=False)
ts1 = pd.to_datetime(ts_str, format="%b %d, %Y %H:%M:%S.%f", errors="coerce")
mask = ts1.isna()
if mask.any():
    ts2 = pd.to_datetime(ts_str[mask], format="%b %d, %Y %H:%M:%S", errors="coerce")
    ts1.loc[mask] = ts2

df_raw["ts"] = ts1
df_raw = df_raw.dropna(subset=["source.ip", "zeek.dns.query", "ts"])

# ---------- 2.5) Pre-filter common benign domains (reduce FP) ----------
# (لیست را در صورت نیاز گسترش بده)
whitelist_patterns = [
    ".google.", ".gstatic.", ".googleusercontent.",
    ".cloudfront.", ".akamai.", ".akamaiedge.", ".edgekey.",
    ".windowsupdate.", ".microsoft.", ".office365.", ".msn.", ".live.",
    ".apple.", ".icloud.",
    ".fbcdn.", ".whatsapp.", ".instagram.", ".facebook.",
    ".telegram.", ".cdn."
]
q_lower = df_raw["zeek.dns.query"].astype(str).str.lower()
mask_whitelist = q_lower.str.contains("|".join(whitelist_patterns), na=False)

# در نسخه 5m-window می‌خواهیم این‌ها روی آموزش اثر نگذارند:
removed_cnt = int(mask_whitelist.sum())
df_raw = df_raw.loc[~mask_whitelist].copy()

# همچنین reverse IPv6 را حذف می‌کنیم تا FP کمتر شود (تمرکز روی IPv4 رفتارها):
mask_v6ptr = q_lower.str.endswith("ip6.arpa", na=False)
removed_v6 = int(mask_v6ptr.sum())
df_raw = df_raw.loc[~mask_v6ptr].copy()

print(f"✅ Pre-filter: removed benign/common={removed_cnt:,} | removed ip6.arpa={removed_v6:,}")

# ---------- 3) Casts ----------
df_raw["zeek.dns.rtt"] = pd.to_numeric(df_raw.get("zeek.dns.rtt"), errors="coerce")
for col in ["zeek.dns.rejected", "zeek.dns.TC"]:
    if col in df_raw.columns:
        s = df_raw[col].astype(str).str.lower()
        df_raw[col] = np.where(s.eq("true"), 1.0, 0.0)

# ---------- 4) Inline lexical features (with DGA-oriented extras) ----------
q_series = df_raw["zeek.dns.query"].astype(str)

df_raw["domain_length"]   = q_series.str.len()
df_raw["subdomain_depth"] = q_series.str.count(r"\.")

def entropy_str_fast(s: str):
    if not s:
        return np.nan
    cnt = np.fromiter(Counter(s).values(), dtype=float)
    p = cnt / cnt.sum()
    return float(-(p * np.log2(p)).sum())

unique_q = pd.unique(q_series)
ent_vals = [entropy_str_fast(s) for s in unique_q]
ent_map  = dict(zip(unique_q, ent_vals))
df_raw["entropy"] = q_series.map(ent_map)

# --- DGA-oriented features ---
_consonant_re = re.compile(r"[bcdfghjklmnpqrstvwxyz]{3,}", re.IGNORECASE)

def consonant_cluster_ratio(q: str) -> float:
    if not isinstance(q, str) or not q:
        return 0.0
    groups = _consonant_re.findall(q)
    return (sum(len(g) for g in groups) / len(q)) if len(q) > 0 else 0.0

def longest_token_len(q: str) -> int:
    if not isinstance(q, str) or not q:
        return 0
    toks = [t for t in q.split(".") if t]
    return max((len(t) for t in toks), default=0)

df_raw["cc_ratio_row"]  = q_series.map(consonant_cluster_ratio)  # consonant cluster ratio
df_raw["ltokenlen_row"] = q_series.map(longest_token_len)        # longest token length

# ---------- 5) TTL/answers (safe parsing for "3,600" etc.) ----------
def _parse_ttls_safe(ttl_str):
    if not isinstance(ttl_str, str):
        return []
    vals = []
    for tok in ttl_str.split(","):
        tok = tok.strip()
        if not tok:
            continue
        tok = tok.replace(",", "")  # remove thousands sep
        try:
            vals.append(float(tok))
        except:
            nums = re.findall(r"\d+", tok)
            if nums:
                try:
                    vals.append(float("".join(nums)))
                except:
                    pass
    return vals

_ttls = df_raw.get("zeek.dns.TTLs")
if _ttls is not None:
    s = _ttls.fillna("").astype(str)
    df_raw["ttl_mean_row"] = s.map(lambda v: (lambda L: np.mean(L) if L else np.nan)(_parse_ttls_safe(v)))
    df_raw["ttl_std_row"]  = s.map(lambda v: (lambda L: np.std (L) if L else np.nan)(_parse_ttls_safe(v)))
else:
    df_raw["ttl_mean_row"] = np.nan
    df_raw["ttl_std_row"]  = np.nan

_ans = df_raw.get("zeek.dns.answers")
if _ans is not None:
    df_raw["answer_count_row"] = _ans.fillna("").astype(str).map(
        lambda s: 0 if not s.strip() else len([x for x in s.split(",") if x.strip()])
    )
else:
    df_raw["answer_count_row"] = 0.0

transport = df_raw.get("network.transport")
df_raw["is_tcp_row"] = (transport.astype(str).str.lower().eq("tcp")).astype(float) if transport is not None else 0.0

# ---------- 6) 5-minute window & aggregation ----------
time_col = "window_5m"  # <— برای سلول‌های بعدی
df_raw[time_col] = df_raw["ts"].dt.floor("5min")
group_keys = ["source.ip", time_col]

agg = df_raw.groupby(group_keys).agg(
    dns_count             = ("zeek.dns.query", "count"),
    unique_domains        = ("zeek.dns.query", pd.Series.nunique),
    nxdomain_ratio        = ("zeek.dns.rcode_name", lambda x: (x == "NXDOMAIN").mean()),
    refused_ratio         = ("zeek.dns.rcode_name", lambda x: (x == "REFUSED").mean()),
    avg_domain_len        = ("domain_length", "mean"),
    avg_entropy           = ("entropy", "mean"),
    avg_subdomain_depth   = ("subdomain_depth", "mean"),
    # DGA-oriented aggregated features:
    avg_cc_ratio          = ("cc_ratio_row", "mean"),
    max_longest_token     = ("ltokenlen_row", "max"),

    avg_rtt               = ("zeek.dns.rtt", "mean"),
    std_rtt               = ("zeek.dns.rtt", "std"),
    avg_ttl               = ("ttl_mean_row", "mean"),
    std_ttl               = ("ttl_std_row", "mean"),
    avg_answer_count      = ("answer_count_row", "mean"),
    tcp_ratio             = ("is_tcp_row", "mean"),
    rejected_ratio        = ("zeek.dns.rejected", "mean"),
    tc_ratio              = ("zeek.dns.TC", "mean"),
).reset_index()

# (اختیاری) فیلتر پنجره‌های خیلی کم‌داده
# agg = agg[agg["dns_count"] >= 5]

feat_cols = [
    "dns_count","unique_domains","nxdomain_ratio","refused_ratio",
    "avg_domain_len","avg_entropy","avg_subdomain_depth",
    "avg_cc_ratio","max_longest_token",
    "avg_rtt","std_rtt","avg_ttl","std_ttl",
    "avg_answer_count","tcp_ratio","rejected_ratio","tc_ratio"
]
agg[feat_cols] = agg[feat_cols].fillna(0.0)

print(f"📦 Windowed feature rows (5m): {len(agg):,}")
print(f"⏱ Prep time: {time.time()-t0:.2f}s")

# ---------- 7) Scale ----------
if_scaler = RobustScaler()   # <— برای استفاده در تست نگه داریم
X = if_scaler.fit_transform(agg[feat_cols].values)

# ---------- 8) Bagged Isolation Forests (optimized; slightly higher contamination) ----------
rng = default_rng(42)
idx_all = np.arange(len(feat_cols))
subA = rng.choice(idx_all, size=int(len(idx_all) * 0.65), replace=False)
subB = rng.choice(idx_all, size=int(len(idx_all) * 0.65), replace=False)

def choose_max_samples(n_rows, frac=0.60, cap=60000):
    return int(min(max(2048, n_rows * frac), cap))

n_rows = X.shape[0]
max_samples_full = choose_max_samples(n_rows, 0.60)
max_samples_sub  = choose_max_samples(n_rows, 0.55)

specs = [
    ("IF_full", idx_all, 400, max_samples_full, 0.008, 17),  # ← contamination بالاتر
    ("IF_subA", subA,     350, max_samples_sub,  0.008, 29),
    ("IF_subB", subB,     350, max_samples_sub,  0.008, 71),
]

scores_list = []
_fitted_models = []  # برای ذخیره مدل‌ها
for name, cols_idx, n_estimators, max_samples, contamination, seed in specs:
    t1 = time.time()
    ifm = IsolationForest(
        n_estimators=n_estimators,
        contamination=contamination,
        max_samples=max_samples,
        random_state=seed,
        n_jobs=-1,
        bootstrap=False
    )
    ifm.fit(X[:, cols_idx])
    s = -ifm.decision_function(X[:, cols_idx])   # larger = more anomalous
    scores_list.append(s)
    _fitted_models.append((name, cols_idx, ifm))  # <— ذخیره
    print(f"🧩 {name} | trees={n_estimators} | max_samples={max_samples} | "
          f"fit+score: {time.time()-t1:.2f}s | mean={s.mean():.4f}")

# آرتیفکت برای سلول‌های بعدی:
if_models = _fitted_models            # list of (name, cols_idx, fitted_IF_model)
if_feat_cols = feat_cols              # برای استفاده آتی
if_time_col  = time_col               # برای exportهای بعدی

scores_mat = np.vstack(scores_list).T
combined = scores_mat.max(axis=1)  # max-fusion: کاهش FN

# ---------- 9) Severity calibration (top 0.5% > 0, top 0.003% = 99) ----------
p_tail = np.percentile(combined, 99.5)    # 0.5%
p_ext  = np.percentile(combined, 99.997)  # 0.003%

sev = np.zeros_like(combined, dtype=float)
lo, hi = p_tail, max(p_ext, p_tail + 1e-12)
mask = combined >= lo
sev[mask] = 1.0 + (combined[mask] - lo) * (98.0 - 1.0) / (hi - lo)
sev[combined >= hi] = 99.0

agg["if_score_combined"] = combined
agg["severity"] = sev
agg["Severity_Label"] = np.where(agg["severity"] >= 99, 99,
                          np.where(agg["severity"] > 0, agg["severity"], 0))

print(f"\n📊 Rows (5m): {len(agg):,} | Sev>0: {(agg['severity']>0).sum()} "
      f"({(agg['severity']>0).mean()*100:.3f}%) | "
      f"Sev=99: {(agg['severity']>=99).sum()} "
      f"({(agg['severity']>=99).mean()*100:.4f}%)")
print(f"⏱ Total Cell 3 runtime: {time.time()-t0:.2f}s")

# ---------- 10) Preview ----------
agg_sorted = agg.sort_values("severity", ascending=False).reset_index(drop=True)
print("\n🔹 5 سطر اول با Severity بالا (۵ دقیقه‌ای):")
display(agg_sorted.head(5))
print("\n🔹 5 سطر آخر با Severity پایین (۵ دقیقه‌ای):")
display(agg_sorted.tail(5))


✅ Pre-filter: removed benign/common=449,529 | removed ip6.arpa=9,103
📦 Windowed feature rows (5m): 20,499
⏱ Prep time: 43.61s
🧩 IF_full | trees=400 | max_samples=12299 | fit+score: 2.63s | mean=-0.1500
🧩 IF_subA | trees=350 | max_samples=11274 | fit+score: 1.88s | mean=-0.1927
🧩 IF_subB | trees=350 | max_samples=11274 | fit+score: 2.22s | mean=-0.1632

📊 Rows (5m): 20,499 | Sev>0: 103 (0.502%) | Sev=99: 1 (0.0049%)
⏱ Total Cell 3 runtime: 50.36s

🔹 5 سطر اول با Severity بالا (۵ دقیقه‌ای):


,source.ip,window_5m,dns_count,unique_domains,nxdomain_ratio,refused_ratio,avg_domain_len,avg_entropy,avg_subdomain_depth,avg_cc_ratio,...,std_rtt,avg_ttl,std_ttl,avg_answer_count,tcp_ratio,rejected_ratio,tc_ratio,if_score_combined,severity,Severity_Label
0,192.168.100.5,2025-09-20 13:20:00,1928,229,0.292531,0.659751,56.090249,4.230818,3.708506,0.092219,...,0.107233,312.214902,267.841240,0.043568,0.0,0.0,0.0,0.202249,99.000000,99.000000
1,10.10.10.41,2025-09-20 13:30:00,378,71,0.121693,0.000000,84.190476,3.773988,11.047619,0.137221,...,1.148581,249.276087,202.987098,1.640212,0.0,0.0,0.0,0.188932,94.628263,94.628263
2,192.168.100.5,2025-10-20 14:35:00,1442,189,0.153953,0.650485,43.876560,3.998612,3.212205,0.080766,...,0.445568,142.648777,127.625488,0.421637,0.0,0.0,0.0,0.181356,89.646265,89.646265
3,192.168.100.119,2025-09-20 13:20:00,782,205,0.693095,0.000000,47.347826,4.037397,3.386189,0.058012,...,0.000371,301.500000,298.500000,0.153453,0.0,0.0,0.0,0.161566,76.632950,76.632950
4,fe80::d3ae:9ee4:f3dd:bd9,2025-09-17 15:05:00,20,2,0.000000,0.000000,57.250000,4.394846,2.750000,0.000000,...,8.755521,56.400000,57.003860,3.500000,0.0,0.0,0.0,0.135529,59.511799,59.511799



🔹 5 سطر آخر با Severity پایین (۵ دقیقه‌ای):


,source.ip,window_5m,dns_count,unique_domains,nxdomain_ratio,refused_ratio,avg_domain_len,avg_entropy,avg_subdomain_depth,avg_cc_ratio,...,std_rtt,avg_ttl,std_ttl,avg_answer_count,tcp_ratio,rejected_ratio,tc_ratio,if_score_combined,severity,Severity_Label
20494,192.168.100.121,2025-10-11 15:40:00,2,1,0.0,0.0,18.000000,3.614369,2.000000,0.000000,...,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,-0.180764,0.0,0.0
20495,192.168.100.121,2025-10-11 15:25:00,2,1,0.0,0.0,18.000000,3.614369,2.000000,0.000000,...,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,-0.180764,0.0,0.0
20496,192.168.100.121,2025-10-11 15:10:00,4,1,0.0,0.0,13.000000,3.392747,2.000000,0.230769,...,0.003464,77.6,111.2,5.00000,0.0,0.0,0.0,-0.155657,0.0,0.0
20497,192.168.100.121,2025-10-11 15:05:00,354,3,0.0,0.0,9.163842,2.737741,0.033898,0.768758,...,0.000000,100.5,99.5,0.00565,0.0,0.0,0.0,-0.070274,0.0,0.0
20498,192.168.100.121,2025-10-12 12:25:00,4,1,0.0,0.0,18.000000,3.614369,2.000000,0.000000,...,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,-0.180764,0.0,0.0


In [4]:
# =======================
# 🔹 Cell 4 — Local Outlier Factor (LOF) Anomaly Detection + SAVE ARTIFACTS
#     - uses same feat_cols from Cell 3 (including DGA features)
#     - novelty=True so we can score on new data
#     - saves artifacts: scaler, model, feat_cols, thresholds, time_col
# =======================
import numpy as np
import pandas as pd
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import RobustScaler
import joblib, time, os

t0 = time.time()

# ---- sanity
if "agg" not in globals() or "feat_cols" not in globals():
    raise RuntimeError("Run Cell 3 first to create `agg` and `feat_cols`.")
time_col = "window_5m" if "window_5m" in agg.columns else ("window_30m" if "window_30m" in agg.columns else None)
if time_col is None:
    raise RuntimeError("Time window column not found in `agg` (window_5m/window_30m).")

df_lof = agg.copy()

# ---- scale
X = df_lof[feat_cols].values
lof_scaler = RobustScaler()
X_scaled = lof_scaler.fit_transform(X)

# ---- dynamic k (پایدارتر روی اندازه‌های مختلف)
def choose_k(n, min_k=20, max_k=60, frac=0.02):
    return int(np.clip(int(n*frac), min_k, max_k))
k = choose_k(len(df_lof), min_k=25, max_k=50, frac=0.02)

lof = LocalOutlierFactor(
    n_neighbors=k,
    contamination=0.008,     # صرفاً تزیینی؛ ما با percentiles خودمان threshold می‌زنیم
    novelty=True,            # تا روی داده جدید score بگیریم
    n_jobs=-1
)

print(f"🧠 آموزش LOF شروع شد ... (n_neighbors={k}, rows={len(df_lof):,})")
lof.fit(X_scaled)
lof_scores = -lof.score_samples(X_scaled)   # بزرگ‌تر ⇒ ناهنجارتر
print(f"✅ آموزش و امتیازدهی LOF انجام شد. ⏱ {time.time()-t0:.2f}s\n")

# ---- severity mapping (هم‌راستا با IF)
TAIL_PCT    = 99.5     # اگر خواستی 0.25% باشه: 99.75
EXTREME_PCT = 99.997
p_tail = np.percentile(lof_scores, TAIL_PCT)
p_ext  = np.percentile(lof_scores, EXTREME_PCT)
lo, hi = p_tail, max(p_ext, p_tail + 1e-12)

sev = np.zeros_like(lof_scores, dtype=float)
mask = lof_scores >= lo
sev[mask] = 1.0 + (lof_scores[mask] - lo) * (98.0 - 1.0) / (hi - lo)
sev[lof_scores >= hi] = 99.0

df_lof["lof_score"] = lof_scores
df_lof["lof_severity"] = np.clip(sev, 0, 99)
df_lof["lof_label"] = (df_lof["lof_severity"] > 0).astype(int)

print(f"📊 LOF rows: {len(df_lof):,} | Sev>0: {(sev>0).sum()} ({(sev>0).mean()*100:.3f}%) | "
      f"Sev=99: {(sev>=99).sum()} ({(sev>=99).mean()*100:.4f}%)")

print("\n🔹 پنج سطر اول LOF با severity بالا:")
display(df_lof.sort_values('lof_severity', ascending=False).head(5))

print("\n🔹 پنج سطر آخر LOF:")
display(df_lof.sort_values('lof_severity', ascending=True).head(5))

# ---- save artifacts for Cell 7
LOF_ARTIFACT_PATH = "/content/lof_artifacts.joblib"
art = {
    "scaler": lof_scaler,
    "model": lof,
    "feat_cols": feat_cols,
    "time_col": time_col,
    "thresholds": {
        "tail_pct": float(TAIL_PCT),
        "extreme_pct": float(EXTREME_PCT),
        "p_tail": float(p_tail),
        "p_ext": float(p_ext)
    }
}
joblib.dump(art, LOF_ARTIFACT_PATH)
print(f"🧰 LOF artifacts saved → {LOF_ARTIFACT_PATH}")


🧠 آموزش LOF شروع شد ... (n_neighbors=50, rows=20,499)
✅ آموزش و امتیازدهی LOF انجام شد. ⏱ 3.94s

📊 LOF rows: 20,499 | Sev>0: 103 (0.502%) | Sev=99: 1 (0.0049%)

🔹 پنج سطر اول LOF با severity بالا:


,source.ip,window_5m,dns_count,unique_domains,nxdomain_ratio,refused_ratio,avg_domain_len,avg_entropy,avg_subdomain_depth,avg_cc_ratio,...,avg_answer_count,tcp_ratio,rejected_ratio,tc_ratio,if_score_combined,severity,Severity_Label,lof_score,lof_severity,lof_label
9855,192.168.100.26,2025-10-20 12:50:00,36,1,0.000000,0.0,4.000000,2.000000,0.000000,0.750000,...,0.000000,0.0,0.0,0.0,0.037004,0.0,0.0,2.666643e+10,99.000000,1
9854,192.168.100.26,2025-10-20 12:35:00,42,3,0.000000,0.0,5.380952,2.181002,0.238095,0.665079,...,0.000000,0.0,0.0,0.0,-0.023197,0.0,0.0,1.965911e+10,84.447851,1
3639,192.168.10.13,2025-10-20 12:40:00,28,3,0.142857,0.0,19.428571,3.250836,1.428571,0.375163,...,0.428571,0.0,0.0,0.0,-0.078051,0.0,0.0,1.827871e+10,77.514625,1
6122,192.168.100.12,2025-10-11 13:50:00,14,2,0.142857,0.0,9.714286,3.179354,0.285714,0.285714,...,0.000000,0.0,0.0,0.0,-0.124662,0.0,0.0,1.789140e+10,75.569354,1
6059,192.168.100.12,2025-10-07 15:25:00,46,4,0.217391,0.0,13.695652,3.252321,0.869565,0.281399,...,0.000000,0.0,0.0,0.0,-0.124715,0.0,0.0,1.650337e+10,68.597835,1



🔹 پنج سطر آخر LOF:


,source.ip,window_5m,dns_count,unique_domains,nxdomain_ratio,refused_ratio,avg_domain_len,avg_entropy,avg_subdomain_depth,avg_cc_ratio,...,avg_answer_count,tcp_ratio,rejected_ratio,tc_ratio,if_score_combined,severity,Severity_Label,lof_score,lof_severity,lof_label
13660,192.168.100.39,2025-10-03 06:20:00,168,3,0.0,0.0,10.714286,2.87255,0.285714,0.688889,...,0.0,0.0,0.0,0.0,-0.130113,0.0,0.0,0.988819,0.0,0
13669,192.168.100.39,2025-10-05 13:50:00,168,3,0.0,0.0,10.714286,2.87255,0.285714,0.688889,...,0.0,0.0,0.0,0.0,-0.130113,0.0,0.0,0.988819,0.0,0
13668,192.168.100.39,2025-10-05 12:20:00,168,3,0.0,0.0,10.714286,2.87255,0.285714,0.688889,...,0.0,0.0,0.0,0.0,-0.130113,0.0,0.0,0.988819,0.0,0
13667,192.168.100.39,2025-10-03 16:50:00,168,3,0.0,0.0,10.714286,2.87255,0.285714,0.688889,...,0.0,0.0,0.0,0.0,-0.130113,0.0,0.0,0.988819,0.0,0
13666,192.168.100.39,2025-10-03 15:20:00,168,3,0.0,0.0,10.714286,2.87255,0.285714,0.688889,...,0.0,0.0,0.0,0.0,-0.130113,0.0,0.0,0.988819,0.0,0


🧰 LOF artifacts saved → /content/lof_artifacts.joblib


In [5]:
# =======================
# 🔹 Cell 5 — Autoencoder (AE) Anomaly Detection (robust tuned) + SAVE ARTIFACTS
# =======================
import os, random, time, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
import joblib

t0 = time.time()

# ---- 0) Reproducibility
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# (اختیاری) جلوگیری از گرفتن کل حافظه GPU
try:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
except Exception:
    pass

# ---- 1) Data & sanity
if "agg" not in globals() or "feat_cols" not in globals():
    raise RuntimeError("Run Cell 3 first (need `agg` & `feat_cols`).")

df_ae = agg.copy()
X = df_ae[feat_cols].values

# ---- 2) Scale
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# (اختیاری) اگر خواستی روی نرمال‌ها تاکید کنی:
# mask_clean = df_ae["severity"] == 0
# X_train_all = X_scaled[mask_clean]
# else:
X_train_all = X_scaled

# Train/Val split
X_tr, X_val = train_test_split(
    X_train_all, test_size=0.2, random_state=SEED, shuffle=True
)
input_dim = X_tr.shape[1]

# ---- 3) Model: undercomplete + regularization
inputs = keras.Input(shape=(input_dim,))
x = layers.Dense(32, activation="relu",
                 kernel_regularizer=regularizers.l2(1e-4))(inputs)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.05)(x)

# Bottleneck smaller than input
x = layers.Dense(8, activation="relu",
                 kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(32, activation="relu",
                 kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.BatchNormalization()(x)

outputs = layers.Dense(input_dim, activation="linear")(x)

autoencoder = keras.Model(inputs, outputs)
autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")

# ---- 4) Callbacks
cbs = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5, verbose=1),
]

autoencoder.summary()

print("🧠 آموزش Autoencoder شروع شد ...")
history = autoencoder.fit(
    X_tr, X_tr,
    validation_data=(X_val, X_val),
    epochs=120,
    batch_size=128,
    shuffle=True,
    callbacks=cbs,
    verbose=1
)
print("✅ آموزش AE تمام شد. ⏱ {:.2f}s\n".format(time.time()-t0))

# ---- 5) Scores on ALL windows
recon = autoencoder.predict(X_scaled, verbose=0)
errors = np.mean(np.square(X_scaled - recon), axis=1)

# ---- 6) Severity mapping (policy-aligned)
# اگر 0.25% می‌خواهی: TAIL_PCT = 99.75
TAIL_PCT    = 99.5
EXTREME_PCT = 99.997

p_tail = np.percentile(errors, TAIL_PCT)
p_ext  = np.percentile(errors, EXTREME_PCT)
lo, hi = p_tail, max(p_ext, p_tail + 1e-12)

sev = np.zeros_like(errors, dtype=float)
mask = errors >= lo
sev[mask] = 1.0 + (errors[mask] - lo) * (98.0 - 1.0) / (hi - lo)
sev[errors >= hi] = 99.0

df_ae["ae_score"] = errors
df_ae["ae_severity"] = np.clip(sev, 0, 99)
df_ae["ae_label"] = (df_ae["ae_severity"] > 0).astype(int)

print(f"📊 AE rows: {len(df_ae):,} | Sev>0: {(sev>0).sum()} ({(sev>0).mean()*100:.3f}%) | "
      f"Sev=99: {(sev>=99).sum()} ({(sev>=99).mean()*100:.4f}%)")
print("\n🔹 پنج سطر اول AE با severity بالا:")
display(df_ae.sort_values('ae_severity', ascending=False).head(5))

# ---- 7) SAVE ARTIFACTS for Cell 7 (test on NEW data)
AE_MODEL_PATH   = "/content/ae_model.keras"          # مدل Keras
AE_SCALER_PATH  = "/content/ae_scaler.joblib"        # اسکیلر
AE_META_PATH    = "/content/ae_meta.joblib"          # متادیتا: feat_cols, thresholds, time_col

# تشخیص ستون پنجره زمانی برای سازگاری با سلول‌های بعدی
time_col = "window_5m" if "window_5m" in df_ae.columns else ("window_30m" if "window_30m" in df_ae.columns else None)

# ذخیره مدل و اسکیلر
autoencoder.save(AE_MODEL_PATH)
joblib.dump(scaler, AE_SCALER_PATH)

# ذخیره متادیتا
meta = {
    "feat_cols": feat_cols,
    "time_col": time_col,
    "thresholds": {
        "tail_pct": float(TAIL_PCT),
        "extreme_pct": float(EXTREME_PCT),
        "p_tail": float(p_tail),
        "p_ext": float(p_ext),
    }
}
joblib.dump(meta, AE_META_PATH)

print(f"🧰 AE artifacts saved →")
print(f"   - model:   {AE_MODEL_PATH}")
print(f"   - scaler:  {AE_SCALER_PATH}")
print(f"   - meta:    {AE_META_PATH}")


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 17)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 8)              │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 17)             │           561 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,977 (7.72 KB)

 Trainable params: 1,833 (7.16 KB)

 Non-trainable params: 144 (576.00 B)

🧠 آموزش Autoencoder شروع شد ...
Epoch 1/120
129/129 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 10.7334 - val_loss: 6.9781 - learning_rate: 0.0010
Epoch 2/120
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4.7069 - val_loss: 3.5562 - learning_rate: 0.0010
Epoch 3/120
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.4393 - val_loss: 1.7699 - learning_rate: 0.0010
Epoch 4/120
129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1.7449 - val_loss: 0.9870 - learning_rate: 0.0010
Epoch 5/120
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.4201 - val_loss: 0.7167 - learning_rate: 0.0010
Epoch 6/120
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.2343 - val_loss: 0.5884 - learning_rate: 0.0010
Epoch 7/120
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.0895 - val_loss: 0.5114 - learning_rate: 0.0010
Epoch 8/120
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9875 - val_loss: 0.4605 - learning_rate: 0.0010
Epoch 9/120
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.9139 - val_loss:

,source.ip,window_5m,dns_count,unique_domains,nxdomain_ratio,refused_ratio,avg_domain_len,avg_entropy,avg_subdomain_depth,avg_cc_ratio,...,avg_answer_count,tcp_ratio,rejected_ratio,tc_ratio,if_score_combined,severity,Severity_Label,ae_score,ae_severity,ae_label
20273,fe80::ed:dcc1:ee65:e5df,2025-09-20 11:20:00,276,3,0.0,0.0,20.913043,3.680909,1.630435,0.048055,...,5.108696,0.0,0.0,0.0,-0.012030,0.000000,0.000000,38.743190,99.000000,1
798,10.10.10.200,2025-09-20 11:20:00,288,5,0.0,0.0,20.805556,3.676430,1.631944,0.049600,...,5.111111,0.0,0.0,0.0,-0.026623,0.000000,0.000000,29.882295,87.259842,1
848,10.10.10.200,2025-09-22 14:00:00,84,4,0.0,0.0,17.785714,3.341513,1.785714,0.161419,...,3.071429,0.0,0.0,0.0,0.023932,0.000000,0.000000,17.868219,49.442144,1
20321,fe80::ed:dcc1:ee65:e5df,2025-09-22 14:00:00,84,4,0.0,0.0,17.785714,3.341513,1.785714,0.161419,...,3.071429,0.0,0.0,0.0,0.023932,0.000000,0.000000,17.854090,49.397671,1
5635,192.168.100.12,2025-09-20 12:25:00,288,4,0.0,0.0,11.916667,2.939487,0.541667,0.644737,...,0.069444,0.0,0.0,0.0,0.120656,49.732189,49.732189,16.300622,44.507689,1


🧰 AE artifacts saved →
   - model:   /content/ae_model.keras
   - scaler:  /content/ae_scaler.joblib
   - meta:    /content/ae_meta.joblib


In [6]:
# =======================
# 🔹 Cell 6 — Minimal, Human-Readable Export (features + per-model severities + final severity) — ROBUST
# =======================
import numpy as np
import pandas as pd
from google.colab import files

# --- sanity checks
need_vars = ["agg", "df_lof", "df_ae", "feat_cols", "df_raw"]
for v in need_vars:
    if v not in globals():
        raise RuntimeError(f"Variable `{v}` not found. Please run Cells 3, 4, 5 first.")

# --- detect time window column (5m or 30m)
time_col = None
for cand in ["window_5m", "window_30m"]:
    if cand in agg.columns:
        time_col = cand
        break
if time_col is None:
    raise RuntimeError("Time window column not found in `agg`. Expected one of: window_5m, window_30m.")
key_cols = ["source.ip", time_col]

# --- ensure df_lof / df_ae also have the time_col
def _ensure_timecol(df_like, df_raw_like, time_col):
    if time_col in df_like.columns:
        return df_like
    # اگر نبود، از df_raw زمان را می‌سازیم و به df_like merge می‌کنیم
    if "ts" not in df_raw_like.columns:
        raise RuntimeError(f"`{time_col}` not in model df and no `ts` in df_raw to derive it.")
    tmp = df_raw_like[["source.ip", "ts"]].copy()
    tmp[time_col] = tmp["ts"].dt.floor("5min" if time_col.endswith("5m") else "30min")
    tmp = tmp.drop(columns=["ts"]).drop_duplicates()
    return df_like.merge(tmp, on="source.ip", how="left")

df_lof = _ensure_timecol(df_lof.copy(), df_raw, time_col)
df_ae  = _ensure_timecol(df_ae.copy(),  df_raw, time_col)

# --- 1) Base features from IF (no model scores)
base_cols = [
    "dns_count","unique_domains","nxdomain_ratio","refused_ratio",
    "avg_domain_len","avg_entropy","avg_subdomain_depth",
    "avg_rtt","std_rtt","avg_ttl","std_ttl",
    "avg_answer_count","tcp_ratio","rejected_ratio","tc_ratio",
]
# فقط ستون‌های موجود
df_base = agg[key_cols + [c for c in base_cols if c in agg.columns]].copy()

# --- 2) Collect severities from each model (ONLY severities)
# IF
df_if_sev = agg[key_cols + ["severity"]].rename(columns={"severity": "if_severity"}).copy()

# LOF
cols_keep_lof = [c for c in ["lof_severity"] if c in df_lof.columns]
df_lof_sev = (df_lof[key_cols + cols_keep_lof].copy()
              if set(key_cols).issubset(df_lof.columns) else
              pd.DataFrame(columns=key_cols + ["lof_severity"]))

# AE
cols_keep_ae = [c for c in ["ae_severity"] if c in df_ae.columns]
df_ae_sev = (df_ae[key_cols + cols_keep_ae].copy()
             if set(key_cols).issubset(df_ae.columns) else
             pd.DataFrame(columns=key_cols + ["ae_severity"]))

# --- 3) Merge to a single table
ens = (df_base
       .merge(df_if_sev, on=key_cols, how="left")
       .merge(df_lof_sev, on=key_cols, how="left")
       .merge(df_ae_sev,  on=key_cols, how="left"))

# fill missing severities with 0
for c in ["if_severity","lof_severity","ae_severity"]:
    if c not in ens.columns:
        ens[c] = 0.0
    ens[c] = ens[c].fillna(0.0)

# --- 4) Compute a simple final severity = max of model severities (like max-fusion)
ens["Final_Severity"] = ens[["if_severity","lof_severity","ae_severity"]].max(axis=1)
ens["Final_Severity_Label"] = np.where(ens["Final_Severity"] >= 99, 99,
                                np.where(ens["Final_Severity"] > 0, ens["Final_Severity"], 0))

# --- 5) Rename columns to human-readable names
rename_map = {
    "source.ip": "Source IP",
    time_col: "Window Start",
    "dns_count": "DNS Query Count",
    "unique_domains": "Unique Domains",
    "nxdomain_ratio": "NXDOMAIN Ratio",
    "refused_ratio": "REFUSED Ratio",
    "avg_domain_len": "Average Domain Length",
    "avg_entropy": "Average Domain Entropy",
    "avg_subdomain_depth": "Average Subdomain Depth",
    "avg_rtt": "Average DNS RTT (s)",
    "std_rtt": "DNS RTT Std (s)",
    "avg_ttl": "Average TTL",
    "std_ttl": "TTL Std",
    "avg_answer_count": "Average Answer Count",
    "tcp_ratio": "TCP Ratio",
    "rejected_ratio": "Rejected Ratio",
    "tc_ratio": "Truncated (TC) Ratio",
    "if_severity": "IF Severity",
    "lof_severity": "LOF Severity",
    "ae_severity": "AE Severity",
    "Final_Severity": "Final Severity",
    "Final_Severity_Label": "Final Severity Label",
}
ens = ens.rename(columns=rename_map)

# --- (optional) round numeric columns for readability
num_cols = [c for c in ens.columns if c not in ["Source IP", "Window Start"]]
for c in num_cols:
    if pd.api.types.is_numeric_dtype(ens[c]):
        ens[c] = ens[c].astype(float).round(6)

# --- 6) Select & order columns (simple features + 3 severities + final)
ordered_cols = [
    "Source IP", "Window Start",
    "DNS Query Count", "Unique Domains",
    "NXDOMAIN Ratio", "REFUSED Ratio",
    "Average Domain Length", "Average Domain Entropy", "Average Subdomain Depth",
    "Average DNS RTT (s)", "DNS RTT Std (s)",
    "Average TTL", "TTL Std",
    "Average Answer Count", "TCP Ratio", "Rejected Ratio", "Truncated (TC) Ratio",
    "IF Severity", "LOF Severity", "AE Severity",
    "Final Severity", "Final Severity Label",
]
ordered_cols = [c for c in ordered_cols if c in ens.columns]
final_out = ens[ordered_cols].copy()

# --- format window as ISO string (readable/consistent)
if "Window Start" in final_out.columns and not pd.api.types.is_string_dtype(final_out["Window Start"]):
    try:
        final_out["Window Start"] = pd.to_datetime(final_out["Window Start"]).dt.strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        pass

# --- fill NaNs in features with 0 for a clean CSV
for c in final_out.columns:
    if pd.api.types.is_numeric_dtype(final_out[c]):
        final_out[c] = final_out[c].fillna(0)

# --- 7) Only alerts where Final Severity > 0 (as requested)
if "Final Severity" in final_out.columns:
    final_alerts = final_out[final_out["Final Severity"] > 0] \
                     .sort_values(["Final Severity","Source IP","Window Start"], ascending=[False, True, True])
else:
    final_alerts = final_out.iloc[0:0]

print(f"📊 Export rows (Final Severity>0): {len(final_alerts):,}")

# --- 8) Save & download
out_path = "/content/anomaly_windows_readable.csv"
# اگر هیچ ردیفی نبود، هدرها ذخیره می‌شوند (CSV خالی ولی استاندارد)
final_alerts.to_csv(out_path, index=False, encoding="utf-8")
print(f"💾 Saved: {out_path}")
files.download(out_path)


📊 Export rows (Final Severity>0): 272
💾 Saved: /content/anomaly_windows_readable.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =======================
# 🔹 Cell 7 — Test on NEW data (5m/30m windows → features → score with existing IF & AE → download)
#      - LOF حذف شده
#      - فیچرهای ساده/خوانا + DGA (avg_cc_ratio/max_longest_token)
#      - اگر آستانه‌های آموزش موجود باشند، همان‌ها استفاده می‌شود (پایدار)، وگرنه Fallback روی داده جدید
# =======================
import os, io, zipfile, re
import numpy as np
import pandas as pd
from collections import Counter
from google.colab import files

# -------------------------------
# 0) Required artifacts (NO TRAINING here)
# -------------------------------
required_vars = {
    "feat_cols": "list of feature column names used in training (Cell 3)",
    "if_scaler": "RobustScaler fitted on training agg (Cell 3)",
    "if_models": "list of tuples (name, cols_idx, fitted_IF_model) from Cell 3 bagging",
    "time_col":  "window column name used in training ('window_5m' or 'window_30m')",
    "ae_scaler": "RobustScaler fitted for AE (Cell 5)",
    "ae_model":  "Keras autoencoder model trained in Cell 5",
}
missing = [k for k in required_vars if k not in globals()]
if missing:
    raise RuntimeError(
        "❌ Missing trained artifacts for TEST-ONLY scoring:\n" +
        "\n".join([f"- {m}: {required_vars[m]}" for m in missing]) +
        "\n\n➡️ لطفاً یک‌بار سلول‌های 3 و 5 را اجرا کنید تا این متغیرها ساخته شوند."
    )

# -------------------------------
# 1) Upload NEW file (CSV or ZIP → first CSV)
# -------------------------------
print("📁 لطفاً فایل CSV یا ZIP را آپلود کنید (همان فرمت قبلی):")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded.")

up_name = next(iter(uploaded.keys()))
raw_bytes = uploaded[up_name]

def load_csv_from_bytes(name: str, data: bytes) -> pd.DataFrame:
    if name.lower().endswith(".zip"):
        z = zipfile.ZipFile(io.BytesIO(data))
        csv_names = [n for n in z.namelist() if n.lower().endswith(".csv")]
        if not csv_names:
            raise RuntimeError("❌ در ZIP هیچ CSV پیدا نشد.")
        with z.open(csv_names[0]) as f:
            return pd.read_csv(f)
    elif name.lower().endswith(".csv"):
        return pd.read_csv(io.BytesIO(data))
    else:
        raise RuntimeError("❌ فرمت پشتیبانی‌نشده. فقط CSV یا ZIP.")

df_new = load_csv_from_bytes(up_name, raw_bytes)
print(f"✅ بارگذاری شد: {len(df_new):,} ردیف")

# -------------------------------
# 2) Minimal cleaning + fast timestamp parse + feature engineering (MIRROR TRAIN)
# -------------------------------
df_raw_new = df_new.copy()

# جایگزینی "-" با NaN
for col in ["zeek.dns.TTLs","zeek.dns.answers","zeek.dns.rtt","zeek.dns.rejected","zeek.dns.TC"]:
    if col in df_raw_new.columns:
        df_raw_new[col] = df_raw_new[col].replace("-", np.nan)

# پارس سریع timestamp (2 مرحله‌ای)
ts_str = df_raw_new["@timestamp"].astype(str).str.replace(" @ ", " ", regex=False)
ts1 = pd.to_datetime(ts_str, format="%b %d, %Y %H:%M:%S.%f", errors="coerce")
mask = ts1.isna()
if mask.any():
    ts2 = pd.to_datetime(ts_str[mask], format="%b %d, %Y %H:%M:%S", errors="coerce")
    ts1.loc[mask] = ts2

df_raw_new["ts"] = ts1
df_raw_new = df_raw_new.dropna(subset=["source.ip","zeek.dns.query","ts"])

# تبدیل انواع
df_raw_new["zeek.dns.rtt"] = pd.to_numeric(df_raw_new.get("zeek.dns.rtt"), errors="coerce")
for col in ["zeek.dns.rejected","zeek.dns.TC"]:
    if col in df_raw_new.columns:
        s = df_raw_new[col].astype(str).str.lower()
        df_raw_new[col] = np.where(s.eq("true"), 1.0, 0.0)

# ===== lexical features (همان آموزش) =====
q_series = df_raw_new["zeek.dns.query"].astype(str)
df_raw_new["domain_length"]   = q_series.str.len()
df_raw_new["subdomain_depth"] = q_series.str.count(r"\.")

def entropy_str_fast(s: str):
    if not s: return np.nan
    cnt = np.fromiter(Counter(s).values(), dtype=float)
    p = cnt / cnt.sum()
    return float(-(p * np.log2(p)).sum())

unique_q = pd.unique(q_series)
ent_vals = [entropy_str_fast(s) for s in unique_q]
ent_map  = dict(zip(unique_q, ent_vals))
df_raw_new["entropy"] = q_series.map(ent_map)

# --- DGA row features (مثل آموزش) ---
_consonant_re = re.compile(r"[bcdfghjklmnpqrstvwxyz]{3,}", re.IGNORECASE)
def consonant_cluster_ratio(q: str) -> float:
    if not isinstance(q, str) or not q:
        return 0.0
    groups = _consonant_re.findall(q)
    return (sum(len(g) for g in groups) / len(q)) if len(q) > 0 else 0.0

def longest_token_len(q: str) -> int:
    if not isinstance(q, str) or not q:
        return 0
    toks = [t for t in q.split(".") if t]
    return max((len(t) for t in toks), default=0)

df_raw_new["cc_ratio_row"]  = q_series.map(consonant_cluster_ratio)
df_raw_new["ltokenlen_row"] = q_series.map(longest_token_len)

# TTL/answers (پارس امن)
def _parse_ttls_safe(ttl_str):
    if not isinstance(ttl_str, str):
        return []
    vals = []
    for tok in ttl_str.split(","):
        tok = tok.strip()
        if not tok: continue
        tok = tok.replace(",", "")
        try:
            vals.append(float(tok))
        except:
            nums = re.findall(r"\d+", tok)
            if nums:
                try: vals.append(float("".join(nums)))
                except: pass
    return vals

_ttls = df_raw_new.get("zeek.dns.TTLs")
if _ttls is not None:
    s = _ttls.fillna("").astype(str)
    df_raw_new["ttl_mean_row"] = s.map(lambda v: (lambda L: np.mean(L) if L else np.nan)(_parse_ttls_safe(v)))
    df_raw_new["ttl_std_row"]  = s.map(lambda v: (lambda L: np.std (L) if L else np.nan)(_parse_ttls_safe(v)))
else:
    df_raw_new["ttl_mean_row"] = np.nan
    df_raw_new["ttl_std_row"]  = np.nan

_ans = df_raw_new.get("zeek.dns.answers")
if _ans is not None:
    df_raw_new["answer_count_row"] = _ans.fillna("").astype(str).map(
        lambda s: 0 if not s.strip() else len([x for x in s.split(",") if x.strip()])
    )
else:
    df_raw_new["answer_count_row"] = 0.0

transport = df_raw_new.get("network.transport")
df_raw_new["is_tcp_row"] = (transport.astype(str).str.lower().eq("tcp")).astype(float) if transport is not None else 0.0

# نام ستون پنجره مثل آموزش
if time_col not in ["window_5m", "window_30m"]:
    raise RuntimeError(f"Unexpected time_col: {time_col}")
df_raw_new[time_col] = df_raw_new["ts"].dt.floor("5min" if time_col.endswith("5m") else "30min")
group_keys = ["source.ip", time_col]

# ===== aggregation (MIRROR TRAIN) =====
agg_new = df_raw_new.groupby(group_keys).agg(
    dns_count             = ("zeek.dns.query","count"),
    unique_domains        = ("zeek.dns.query", pd.Series.nunique),
    nxdomain_ratio        = ("zeek.dns.rcode_name", lambda x: (x=="NXDOMAIN").mean()),
    refused_ratio         = ("zeek.dns.rcode_name", lambda x: (x=="REFUSED").mean()),
    avg_domain_len        = ("domain_length","mean"),
    avg_entropy           = ("entropy","mean"),
    avg_subdomain_depth   = ("subdomain_depth","mean"),
    # DGA aggregated features:
    avg_cc_ratio          = ("cc_ratio_row", "mean"),
    max_longest_token     = ("ltokenlen_row", "max"),

    avg_rtt               = ("zeek.dns.rtt","mean"),
    std_rtt               = ("zeek.dns.rtt","std"),
    avg_ttl               = ("ttl_mean_row","mean"),
    std_ttl               = ("ttl_std_row","mean"),
    avg_answer_count      = ("answer_count_row","mean"),
    tcp_ratio             = ("is_tcp_row","mean"),
    rejected_ratio        = ("zeek.dns.rejected","mean"),
    tc_ratio              = ("zeek.dns.TC","mean"),
).reset_index()

# --- فیچرهای خواناتر (مشتق ساده) ---
agg_new["qpm"] = agg_new["dns_count"] / 5.0
agg_new["qps"] = agg_new["dns_count"] / 300.0
agg_new["unique_domain_ratio"] = np.where(
    agg_new["dns_count"] > 0,
    agg_new["unique_domains"] / agg_new["dns_count"],
    0.0
)
err_sum = agg_new["nxdomain_ratio"].fillna(0) + agg_new["refused_ratio"].fillna(0)
agg_new["error_ratio"] = err_sum.clip(lower=0.0, upper=1.0)
agg_new["udp_ratio"] = 1.0 - agg_new["tcp_ratio"].fillna(0.0)

# Top-1 Domain / Resolver
vc_domains = (df_raw_new.groupby(group_keys + ["zeek.dns.query"])
              .size().reset_index(name="cnt"))
top1d = (vc_domains.sort_values(["source.ip", time_col, "cnt"], ascending=[True, True, False])
                  .groupby(group_keys).head(1))
top1d["Top Domain"] = top1d["zeek.dns.query"].astype(str) + " (" + top1d["cnt"].astype(int).astype(str) + ")"
top1d = top1d[group_keys + ["Top Domain"]]

vc_res = (df_raw_new.groupby(group_keys + ["destination.ip"])
          .size().reset_index(name="cnt"))
top1r = (vc_res.sort_values(["source.ip", time_col, "cnt"], ascending=[True, True, False])
                 .groupby(group_keys).head(1))
top1r["Top Resolver"] = top1r["destination.ip"].astype(str) + " (" + top1r["cnt"].astype(int).astype(str) + ")"
top1r = top1r[group_keys + ["Top Resolver"]]

agg_new = (agg_new.merge(top1d, on=group_keys, how="left")
                  .merge(top1r, on=group_keys, how="left"))

# هم‌ترازسازی فیچرها با آموزش
for c in feat_cols:
    if c not in agg_new.columns:
        agg_new[c] = 0.0
agg_new[feat_cols] = agg_new[feat_cols].fillna(0.0)

# -------------------------------
# 3) SCORE with existing IF & AE (NO LOF)
#    - thresholds: prefer TRAIN thresholds if available (stable); else fallback on NEW data percentiles
# -------------------------------

# ===== IF (bagged) =====
X_if = if_scaler.transform(agg_new[feat_cols].values)
if_scores_list = []
for (name, cols_idx, ifm) in if_models:
    s = -ifm.decision_function(X_if[:, cols_idx])
    if_scores_list.append(s)
if_scores = np.vstack(if_scores_list).T if if_scores_list else np.zeros((len(agg_new), 0))
agg_new["if_severity"] = 0.0

if if_scores.size:
    combined = if_scores.max(axis=1)  # max-fusion

    # آستانه‌های آموزش اگر حاضر باشند (از Cell 3)
    if ("agg" in globals()) and ("if_score_combined" in agg.columns):
        p_tail_train = np.percentile(agg["if_score_combined"].values, 99.5)
        p_ext_train  = np.percentile(agg["if_score_combined"].values, 99.997)
        lo, hi = p_tail_train, max(p_ext_train, p_tail_train + 1e-12)
    else:
        # fallback روی داده‌ی جدید
        p_tail_new = np.percentile(combined, 99.5)
        p_ext_new  = np.percentile(combined, 99.997)
        lo, hi = p_tail_new, max(p_ext_new, p_tail_new + 1e-12)

    sev = np.zeros_like(combined, dtype=float)
    mask = combined >= lo
    sev[mask] = 1.0 + (combined[mask] - lo) * (98.0 - 1.0) / (hi - lo)
    sev[combined >= hi] = 99.0
    agg_new["if_severity"] = np.clip(sev, 0, 99)

# ===== AE =====
X_ae = ae_scaler.transform(agg_new[feat_cols].values)
recon = ae_model.predict(X_ae, verbose=0)
errors = np.mean(np.square(X_ae - recon), axis=1)

# آستانه‌های آموزش اگر حاضر باشند (از Cell 5)
if ("df_ae" in globals()) and ("ae_score" in df_ae.columns):
    p_tail_train = np.percentile(df_ae["ae_score"].values, 99.5)
    p_ext_train  = np.percentile(df_ae["ae_score"].values, 99.997)
    lo, hi = p_tail_train, max(p_ext_train, p_tail_train + 1e-12)
else:
    # fallback روی داده‌ی جدید
    p_tail_new = np.percentile(errors, 99.5)
    p_ext_new  = np.percentile(errors, 99.997)
    lo, hi = p_tail_new, max(p_ext_new, p_tail_new + 1e-12)

sev = np.zeros_like(errors, dtype=float)
mask = errors >= lo
sev[mask] = 1.0 + (errors[mask] - lo) * (98.0 - 1.0) / (hi - lo)
sev[errors >= hi] = 99.0
agg_new["ae_severity"] = np.clip(sev, 0, 99)

# Final severity
agg_new["Final_Severity"] = agg_new[["if_severity","ae_severity"]].max(axis=1)
agg_new["Final_Severity_Label"] = np.where(agg_new["Final_Severity"] >= 99, 99,
                                    np.where(agg_new["Final_Severity"] > 0, agg_new["Final_Severity"], 0))

# -------------------------------
# 4) Human-readable export (simple features + derived + IF/AE + final)
# -------------------------------
rename_map = {
    "source.ip": "Source IP",
    time_col: "Window Start",
    # پایه
    "dns_count": "DNS Query Count",
    "unique_domains": "Unique Domains",
    "nxdomain_ratio": "NXDOMAIN Ratio",
    "refused_ratio": "REFUSED Ratio",
    "avg_domain_len": "Average Domain Length",
    "avg_entropy": "Average Domain Entropy",
    "avg_subdomain_depth": "Average Subdomain Depth",
    "avg_rtt": "Average DNS RTT (s)",
    "std_rtt": "DNS RTT Std (s)",
    "avg_ttl": "Average TTL",
    "std_ttl": "TTL Std",
    "avg_answer_count": "Average Answer Count",
    "tcp_ratio": "TCP Ratio",
    "rejected_ratio": "Rejected Ratio",
    "tc_ratio": "Truncated (TC) Ratio",
    # فیچرهای DGA (درصورت نیاز می‌توانی از خروجی حذفشان کنی)
    "avg_cc_ratio": "Consonant Cluster Ratio (avg)",
    "max_longest_token": "Longest Token Length (max)",
    # مشتق ساده
    "qpm": "Queries per Minute",
    "qps": "Queries per Second",
    "unique_domain_ratio": "Unique Domain Ratio",
    "error_ratio": "Total Error Ratio",
    "udp_ratio": "UDP Ratio",
    # مدل‌ها
    "if_severity": "IF Severity",
    "ae_severity": "AE Severity",
    "Final_Severity": "Final Severity",
    "Final_Severity_Label": "Final Severity Label",
}
out_df = agg_new.rename(columns=rename_map)

# ترتیب ستون‌ها
ordered_cols = [
    "Source IP", "Window Start",
    "Top Domain", "Top Resolver",
    "DNS Query Count", "Queries per Minute", "Queries per Second",
    "Unique Domains", "Unique Domain Ratio",
    "NXDOMAIN Ratio", "REFUSED Ratio", "Total Error Ratio",
    "Average Domain Length", "Average Domain Entropy", "Average Subdomain Depth",
    "Consonant Cluster Ratio (avg)", "Longest Token Length (max)",
    "Average DNS RTT (s)", "DNS RTT Std (s)",
    "Average TTL", "TTL Std",
    "Average Answer Count", "TCP Ratio", "UDP Ratio", "Rejected Ratio", "Truncated (TC) Ratio",
    "IF Severity", "AE Severity",
    "Final Severity", "Final Severity Label",
]
ordered_cols = [c for c in ordered_cols if c in out_df.columns]
final_out = out_df[ordered_cols].copy()

# فرمت ستون زمان
if "Window Start" in final_out.columns and not pd.api.types.is_string_dtype(final_out["Window Start"]):
    try:
        final_out["Window Start"] = pd.to_datetime(final_out["Window Start"]).dt.strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        pass

# پاکسازی NaN های عددی برای CSV تمیز
for c in final_out.columns:
    if pd.api.types.is_numeric_dtype(final_out[c]):
        final_out[c] = final_out[c].fillna(0)

# فقط هشدارها (Final Severity > 0)
if "Final Severity" in final_out.columns:
    final_alerts = final_out[final_out["Final Severity"] > 0] \
                     .sort_values(["Final Severity","Source IP","Window Start"], ascending=[False, True, True])
else:
    final_alerts = final_out.iloc[0:0]

print(f"📊 Test windows: {len(final_out):,} | Alerts (Final Severity>0): {len(final_alerts):,}")

# ذخیره و دانلود
out_path = "/content/anomaly_windows_readable_TEST.csv"
final_alerts.to_csv(out_path, index=False, encoding="utf-8")
print(f"💾 Saved: {out_path}")
files.download(out_path)


📁 لطفاً فایل CSV یا ZIP را آپلود کنید (همان فرمت قبلی):


Saving Untitled Discover session (31).zip to Untitled Discover session (31) (2).zip
✅ بارگذاری شد: 144,500 ردیف
📊 Test windows: 1,536 | Alerts (Final Severity>0): 12
💾 Saved: /content/anomaly_windows_readable_TEST.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# =======================
# 🔹 Cell 8 — Train dual IF on RAW rows (IPv4/IPv6 split, no aggregation)
#      - ONLY zeek.dns.query-based features (row-wise, fast via unique-cache)
#      - Split: IPv4 = not ip6.arpa, IPv6 = ip6.arpa
#      - Local bucket ('.local', no-dot, GUID.local) ⇒ excluded (severity=0)
#      - Severity policy (per split, NON-LOCAL rows only):
#           top 0.5%  ⇒ Severity > 0
#           top 0.003% ⇒ Severity = 99
#      - Saves artifacts for both splits in one file
# =======================
import os, io, zipfile, time, re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from google.colab import files

# -------- Config --------
ARTIFACT_PATH = "/content/query_if_artifacts_dual.joblib"  # <— یکسان با سلول 9
OUT_V4 = "/content/dns_query_IF_train_anomalies_RAW_v4.csv"
OUT_V6 = "/content/dns_query_IF_train_anomalies_RAW_v6.csv"

# Percentiles (per split, NON-LOCAL rows)
TAIL_PCT    = 99.5     # top 0.5%  → Severity > 0
EXTREME_PCT = 99.997   # top 0.003% → Severity = 99

# Prefilter (remove obvious noise-only)
MIN_LEN = 8

# Regexes
RE_GUID_LOCAL  = re.compile(r'^[0-9a-f]{8}-(?:[0-9a-f]{4}-){3}[0-9a-f]{12}\.local$', re.IGNORECASE)
RE_IPV6_PTR    = re.compile(r'^(?:[0-9a-f]\.){32}ip6\.arpa$', re.IGNORECASE)

# ---------- Helpers ----------
def extract_tld(q: str) -> str:
    if not isinstance(q, str) or not q: return ""
    parts = q.strip(".").split(".")
    return parts[-1].lower() if parts else ""

def entropy_str(s: str) -> float:
    if not s: return 0.0
    freq = np.fromiter(Counter(s).values(), dtype=float)
    p = freq / freq.sum()
    return float(-(p * np.log2(p)).sum())

def avg_token_length(s: str) -> float:
    toks = [t for t in s.split(".") if t]
    return float(np.mean([len(t) for t in toks])) if toks else float(len(s))

def max_token_length(s: str) -> float:
    toks = [t for t in s.split(".") if t]
    return float(max([len(t) for t in toks])) if toks else float(len(s))

def vowel_ratio(s: str) -> float:
    if not s: return 0.0
    v = set("aeiou")
    return sum(ch in v for ch in s) / len(s)

def consonant_ratio(s: str) -> float:
    if not s: return 0.0
    letters = [c for c in s if c.isalpha()]
    if not letters: return 0.0
    return sum(c not in "aeiou" for c in letters) / len(letters)

def nonalpha_ratio(s: str) -> float:
    if not s: return 0.0
    # exclude dot and hyphen from nonalpha count
    return sum((not c.isalnum()) and (c not in ".-") for c in s) / len(s)

def bigram_diversity(s: str) -> float:
    # diversity = unique bigrams / possible bigrams
    if not s or len(s) < 2: return 0.0
    bgs = set(s[i:i+2] for i in range(len(s)-1))
    return len(bgs) / float(len(s)-1)

def build_features_raw(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Fast row-wise features using unique-cache mapping."""
    feat_df = pd.DataFrame({
        "@timestamp": df_raw.get("@timestamp", pd.Series([""]*len(df_raw))),
        "source.ip": df_raw["source.ip"].astype(str),
        "zeek.dns.query": df_raw["zeek.dns.query"].astype(str),
    })

    # lowercase once (robust to case)
    q = feat_df["zeek.dns.query"].str.strip().str.lower()
    feat_df["zeek.dns.query"] = q

    # unique-cache for heavy funcs
    uq = pd.unique(q)

    ent_map    = {s: entropy_str(s)        for s in uq}
    avglen_map = {s: avg_token_length(s)   for s in uq}
    maxlen_map = {s: max_token_length(s)   for s in uq}
    vow_map    = {s: vowel_ratio(s)        for s in uq}
    cons_map   = {s: consonant_ratio(s)    for s in uq}
    tld_map    = {s: extract_tld(s)        for s in uq}
    digit_map  = {s: (sum(c.isdigit() for c in s) / len(s) if len(s)>0 else 0.0) for s in uq}
    nonalp_map = {s: nonalpha_ratio(s)     for s in uq}
    bdiv_map   = {s: bigram_diversity(s)   for s in uq}

    feat_df["query_length"]    = q.str.len()
    feat_df["dot_count"]       = q.str.count(r"\.")
    feat_df["digit_ratio"]     = q.map(digit_map)
    feat_df["entropy"]         = q.map(ent_map)
    feat_df["has_hyphen"]      = q.str.contains("-", regex=False).astype(int)
    feat_df["avg_token_len"]   = q.map(avglen_map)
    feat_df["max_token_len"]   = q.map(maxlen_map)
    feat_df["vowel_ratio"]     = q.map(vow_map)
    feat_df["consonant_ratio"] = q.map(cons_map)
    feat_df["nonalpha_ratio"]  = q.map(nonalp_map)
    feat_df["bigram_div"]      = q.map(bdiv_map)
    feat_df["tld"]             = q.map(tld_map)
    feat_df["is_local"]        = feat_df["tld"].eq("local").astype(int)
    feat_df["is_guid_local"]   = q.str.match(RE_GUID_LOCAL).astype(int)
    feat_df["is_ip6_ptr"]      = q.str.match(RE_IPV6_PTR).astype(int)

    # Reason (برای گزارش)
    reason = np.where(feat_df["is_guid_local"]==1, "GUID.local",
              np.where(feat_df["is_ip6_ptr"]==1, "IPv6 PTR",
              np.where(feat_df["is_local"]==1, "mDNS/.local",
              np.where(feat_df["dot_count"]==0, "single-label", ""))))
    feat_df["Reason"] = reason

    # Local bucket → برای حذف از آموزش/امتیاز دهی
    feat_df["is_local_bucket"] = (
        (feat_df["is_local"]==1) |
        (feat_df["dot_count"]==0) |
        (feat_df["is_guid_local"]==1)
    ).astype(int)

    return feat_df

def train_split_raw(name: str, df_split: pd.DataFrame):
    """Train IF on RAW rows for one split (v4/v6). Excludes local bucket from training/scoring."""
    if df_split is None or len(df_split)==0:
        return None, None, None, None

    feat_df = build_features_raw(df_split)

    # فیچرهای مدل (شامل فیچرهای جدید)
    feat_cols = [
        "query_length","dot_count","digit_ratio","entropy",
        "has_hyphen","avg_token_len","max_token_len",
        "vowel_ratio","consonant_ratio","nonalpha_ratio","bigram_div",
        "is_local","is_guid_local"
    ]

    # NON-LOCAL rows for training/scoring
    mask_nonlocal = (feat_df["is_local_bucket"]==0)
    nonlocal_df = feat_df.loc[mask_nonlocal].copy()

    if len(nonlocal_df) == 0:
        print(f"⚠️ {name}: No NON-LOCAL rows; skipping training.")
        feat_df["if_row_score"] = 0.0
        feat_df["if_severity"]  = 0.0
        anoms = feat_df[feat_df["if_severity"] > 0].copy()
        return None, anoms, feat_df, feat_cols

    # Scale
    scaler = RobustScaler()
    X = scaler.fit_transform(nonlocal_df[feat_cols].values)

    # Sample weight (downweight frequent queries)
    counts = nonlocal_df["zeek.dns.query"].value_counts()
    base_w = nonlocal_df["zeek.dns.query"].map(lambda s: 1.0 / min(counts[s], 50)).values

    if_model = IsolationForest(
        n_estimators=400,
        contamination=0.001,   # hint only
        max_samples="auto",
        random_state=42,
        n_jobs=-1
    )
    print(f"🧠 Training IF — {name} (RAW, NON-LOCAL) | rows={len(nonlocal_df):,}")
    if_model.fit(X, sample_weight=base_w)

    # Score ONLY on NON-LOCAL rows
    scores = -if_model.decision_function(X)
    nonlocal_df["if_row_score"] = scores

    # Severity mapping (NON-LOCAL rows)
    p_tail = np.percentile(scores, TAIL_PCT)
    p_ext  = np.percentile(scores, EXTREME_PCT)
    lo, hi = p_tail, max(p_ext, p_tail + 1e-12)

    sev = np.zeros(len(scores), dtype=float)
    m = scores >= lo
    sev[m] = 1.0 + (scores[m] - lo) * (98.0 - 1.0) / (hi - lo)
    sev[scores >= hi] = 99.0
    nonlocal_df["if_severity"] = np.clip(sev, 0, 99)

    # Merge back to full feat_df; local bucket = 0
    feat_df = feat_df.merge(
        nonlocal_df[["zeek.dns.query","source.ip","@timestamp","if_row_score","if_severity"]],
        on=["zeek.dns.query","source.ip","@timestamp"], how="left"
    )
    feat_df["if_row_score"] = feat_df["if_row_score"].fillna(0.0)
    feat_df["if_severity"]  = feat_df["if_severity"].fillna(0.0)

    # anomalies (only NON-LOCAL with Sev>0)
    anoms = feat_df[(feat_df["is_local_bucket"]==0) & (feat_df["if_severity"] > 0)].copy()
    anoms.insert(1, "Query_Type", "IPv6" if name=="IPv6" else "IPv4")

    # گزارش حجم باکت محلی
    local_cnt = int((feat_df["is_local_bucket"]==1).sum())
    print(f"📊 {name}: NON-LOCAL rows={len(nonlocal_df):,} | LOCAL bucket skipped={local_cnt:,}")
    print(f"📈 {name}: p_tail={p_tail:.6f} | p_ext={p_ext:.6f} | anomalies~0.5%={len(anoms):,}")

    artifacts = {
        "scaler": scaler,
        "if_model": if_model,
        "feat_cols": feat_cols,
        "policy": {
            "scope": "raw_rows_nonlocal",
            "tail_pct": float(TAIL_PCT),
            "extreme_pct": float(EXTREME_PCT),
            "p_tail": float(p_tail),
            "p_ext": float(p_ext)
        }
    }
    return artifacts, anoms, feat_df, feat_cols

# ---------- Load input (ZIP/CSV) ----------
t0 = time.time()

default_candidates = [
    "/content/merged (3)dns.zip",
    "/content/merged (3).zip",
    "/content/zeek_data/merged (3).csv",
]
df_raw = None

def _read_csv_from_zip_bytes(raw_bytes):
    zf = zipfile.ZipFile(io.BytesIO(raw_bytes))
    csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
    if not csv_names:
        raise RuntimeError("❌ در فایل ZIP هیچ CSV پیدا نشد.")
    with zf.open(csv_names[0]) as f:
        return pd.read_csv(f, sep=None, engine="python")

def _load_from_path(path):
    if path.lower().endswith(".zip"):
        with open(path, "rb") as f:
            return _read_csv_from_zip_bytes(f.read())
    elif path.lower().endswith(".csv"):
        return pd.read_csv(path, sep=None, engine="python")
    else:
        return None

for p in default_candidates:
    if os.path.exists(p):
        try:
            df_raw = _load_from_path(p)
            if df_raw is not None:
                print(f"✅ Loaded from: {p}  | rows={len(df_raw):,}")
                break
        except Exception as e:
            print(f"⚠️ Could not load {p}: {e}")

if df_raw is None:
    print("📁 لطفاً فایل ZIP/CSV ورودی خود را آپلود کنید (مثل: merged (3)dns.zip):")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("❌ No file uploaded.")
    name, raw_bytes = next(iter(uploaded.items()))
    if name.lower().endswith(".zip"):
        df_raw = _read_csv_from_zip_bytes(raw_bytes)
        print(f"✅ Loaded ZIP → CSV rows={len(df_raw):,}")
    elif name.lower().endswith(".csv"):
        df_raw = pd.read_csv(io.BytesIO(raw_bytes), sep=None, engine="python")
        print(f"✅ Loaded CSV rows={len(df_raw):,}")
    else:
        raise RuntimeError("❌ فرمت پشتیبانی‌نشده (فقط ZIP یا CSV).")

# ---------- Cleanup & minimal noise filter ----------
need_cols = ["source.ip","zeek.dns.query"]
for c in need_cols:
    if c not in df_raw.columns:
        raise RuntimeError(f"❌ ستون لازم یافت نشد: {c}")

df_raw = df_raw.dropna(subset=["zeek.dns.query"]).copy()
df_raw["source.ip"] = df_raw["source.ip"].astype(str)
df_raw["zeek.dns.query"] = df_raw["zeek.dns.query"].astype(str).str.strip().str.lower()

mask_valid = df_raw["zeek.dns.query"].str.len() >= MIN_LEN
mask_valid &= ~df_raw["zeek.dns.query"].eq("-")
df_raw = df_raw.loc[mask_valid].copy()
print(f"🧹 Prefilter done. RAW rows={len(df_raw):,}")

# ---------- Split by ip6.arpa ----------
is_v6 = df_raw["zeek.dns.query"].str.endswith("ip6.arpa", na=False)
df_v6 = df_raw[is_v6].copy()
df_v4 = df_raw[~is_v6].copy()
print(f"🔀 Split → IPv4 rows={len(df_v4):,} | IPv6 rows={len(df_v6):,}")

# ---------- Train per split (RAW, NON-LOCAL) ----------
art_v4, anoms_v4, rows_v4, feat_cols_v4 = train_split_raw("IPv4", df_v4)
art_v6, anoms_v6, rows_v6, feat_cols_v6 = train_split_raw("IPv6", df_v6)

# ---------- Save anomalies CSVs (only NON-LOCAL anomalies) ----------
def _export(anoms, path):
    if anoms is None or len(anoms)==0:
        print(f"ℹ️ No anomalies to save for: {path}")
        return
    cols_out = [
        "@timestamp","source.ip","zeek.dns.query","Query_Type","Reason",
        "if_row_score","if_severity",
        # features (for triage/readability)
        "query_length","dot_count","digit_ratio","entropy",
        "has_hyphen","avg_token_len","max_token_len","vowel_ratio","consonant_ratio",
        "nonalpha_ratio","bigram_div",
        "tld","is_local","is_guid_local","is_ip6_ptr","is_local_bucket"
    ]
    exist = [c for c in cols_out if c in anoms.columns]
    anoms.sort_values("if_row_score", ascending=False)[exist].to_csv(path, index=False, encoding="utf-8")
    print(f"💾 Saved: {path}")
    files.download(path)

if anoms_v4 is not None: _export(anoms_v4, OUT_V4)
if anoms_v6 is not None: _export(anoms_v6, OUT_V6)

# ---------- Save artifacts ----------
try:
    import joblib
    joblib.dump({
        "v4": {"scaler": art_v4["scaler"], "if_model": art_v4["if_model"], "feat_cols": art_v4["feat_cols"], "policy": art_v4["policy"]} if art_v4 else None,
        "v6": {"scaler": art_v6["scaler"], "if_model": art_v6["if_model"], "feat_cols": art_v6["feat_cols"], "policy": art_v6["policy"]} if art_v6 else None,
    }, ARTIFACT_PATH)
    print(f"🧰 Artifacts saved (dual RAW, NON-LOCAL only) → {ARTIFACT_PATH}")
except Exception as e:
    print(f"⚠️ Could not save artifacts: {e}")

print(f"⏱ Total runtime: {time.time()-t0:.2f}s")


✅ Loaded from: /content/merged (3)dns.zip  | rows=2,236,320
🧹 Prefilter done. RAW rows=2,191,971
🔀 Split → IPv4 rows=2,182,868 | IPv6 rows=9,103
🧠 Training IF — IPv4 (RAW, NON-LOCAL) | rows=896,057
📊 IPv4: NON-LOCAL rows=896,057 | LOCAL bucket skipped=1,286,811
📈 IPv4: p_tail=-0.018664 | p_ext=0.028432 | anomalies~0.5%=9,291
🧠 Training IF — IPv6 (RAW, NON-LOCAL) | rows=9,103
📊 IPv6: NON-LOCAL rows=9,103 | LOCAL bucket skipped=0
📈 IPv6: p_tail=0.000000 | p_ext=-0.000000 | anomalies~0.5%=755
💾 Saved: /content/dns_query_IF_train_anomalies_RAW_v4.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

💾 Saved: /content/dns_query_IF_train_anomalies_RAW_v6.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🧰 Artifacts saved (dual RAW, NON-LOCAL only) → /content/query_if_artifacts_dual.joblib
⏱ Total runtime: 55.77s


In [9]:
# =======================
# 🔹 Cell 9 — Inference ONLY (dual models; IPv4 & IPv6 split)
#      - NO training
#      - Uses Cell 8 artifacts (scaler/model) فقط برای score
#      - Severity mapping روی دیتای جدید: top 0.5% ⇒ >0 ، top 0.003% ⇒ 99
#      - LOCAL bucket (.local / single-label / GUID.local) ⇒ severity=0
#      - Optimized with unique-cache for features
# =======================
import os, io, zipfile, time, re, hashlib, datetime
import numpy as np
import pandas as pd
from collections import Counter
from google.colab import files

ARTIFACT_PATH = "/content/query_if_artifacts_dual.joblib"
MIN_LEN = 8  # همراستا با سلول 8

# --- Regex & helpers (مطابق سلول 8)
RE_GUID_LOCAL  = re.compile(r'^[0-9a-f]{8}-(?:[0-9a-f]{4}-){3}[0-9a-f]{12}\.local$', re.IGNORECASE)
RE_IPV6_PTR    = re.compile(r'^(?:[0-9a-f]\.){32}ip6\.arpa$', re.IGNORECASE)

def extract_tld(q: str) -> str:
    if not isinstance(q, str) or not q: return ""
    parts = q.strip(".").split(".")
    return parts[-1].lower() if parts else ""

def entropy_str(s: str) -> float:
    if not s: return 0.0
    freq = np.fromiter(Counter(s).values(), dtype=float)
    p = freq / freq.sum()
    return float(-(p * np.log2(p)).sum())

def avg_token_length(s: str) -> float:
    toks = [t for t in s.split(".") if t]
    return float(np.mean([len(t) for t in toks])) if toks else float(len(s))

def max_token_length(s: str) -> float:
    toks = [t for t in s.split(".") if t]
    return float(max([len(t) for t in toks])) if toks else float(len(s))

def vowel_ratio(s: str) -> float:
    if not s: return 0.0
    v = set("aeiou")
    return sum(ch in v for ch in s) / len(s)

def consonant_ratio(s: str) -> float:
    if not s: return 0.0
    letters = [c for c in s if c.isalpha()]
    if not letters: return 0.0
    return sum(c not in "aeiou" for c in letters) / len(letters)

def nonalpha_ratio(s: str) -> float:
    if not s: return 0.0
    return sum((not c.isalnum()) and (c not in ".-") for c in s) / len(s)

def bigram_diversity(s: str) -> float:
    if not s or len(s) < 2: return 0.0
    bgs = set(s[i:i+2] for i in range(len(s)-1))
    return len(bgs) / float(len(s)-1)

def build_features_cached(df_raw: pd.DataFrame) -> pd.DataFrame:
    """همان فیچرهای سلول 8، با کش روی queries یکتا برای سرعت."""
    feat_df = pd.DataFrame({
        "source.ip": df_raw["source.ip"].astype(str),
        "zeek.dns.query": df_raw["zeek.dns.query"].astype(str)
    })
    q = feat_df["zeek.dns.query"].str.strip().str.lower()
    feat_df["zeek.dns.query"] = q

    uq = pd.unique(q)
    print(f"🔧 Building cached features on {len(uq):,} unique queries ...")

    ent_map    = {s: entropy_str(s)        for s in uq}
    avglen_map = {s: avg_token_length(s)   for s in uq}
    maxlen_map = {s: max_token_length(s)   for s in uq}
    vow_map    = {s: vowel_ratio(s)        for s in uq}
    cons_map   = {s: consonant_ratio(s)    for s in uq}
    tld_map    = {s: extract_tld(s)        for s in uq}
    digit_map  = {s: (sum(c.isdigit() for c in s) / len(s) if len(s)>0 else 0.0) for s in uq}
    nonalp_map = {s: nonalpha_ratio(s)     for s in uq}
    bdiv_map   = {s: bigram_diversity(s)   for s in uq}

    feat_df["query_length"]    = q.str.len()
    feat_df["dot_count"]       = q.str.count(r"\.")
    feat_df["digit_ratio"]     = q.map(digit_map)
    feat_df["entropy"]         = q.map(ent_map)
    feat_df["has_hyphen"]      = q.str.contains("-", regex=False).astype(int)
    feat_df["avg_token_len"]   = q.map(avglen_map)
    feat_df["max_token_len"]   = q.map(maxlen_map)
    feat_df["vowel_ratio"]     = q.map(vow_map)
    feat_df["consonant_ratio"] = q.map(cons_map)
    feat_df["nonalpha_ratio"]  = q.map(nonalp_map)
    feat_df["bigram_div"]      = q.map(bdiv_map)
    feat_df["tld"]             = q.map(tld_map)
    feat_df["is_local"]        = feat_df["tld"].eq("local").astype(int)
    feat_df["is_guid_local"]   = q.str.match(RE_GUID_LOCAL).astype(int)
    feat_df["is_ip6_ptr"]      = q.str.match(RE_IPV6_PTR).astype(int)

    feat_df["is_local_bucket"] = (
        (feat_df["is_local"]==1) |
        (feat_df["dot_count"]==0) |
        (feat_df["is_guid_local"]==1)
    ).astype(int)
    return feat_df

def map_severity_percentile_on_new(scores: np.ndarray) -> np.ndarray:
    """
    نگاشت severity روی دیتای جدید:
      - top 0.5% ⇒ >0
      - top 0.003% ⇒ 99
    مراقبت برای دیتاست کوچک و آستانه‌های برابر.
    """
    n = len(scores)
    if n == 0:
        return np.zeros(0, dtype=float)

    lo = np.percentile(scores, 99.)
    hi = np.percentile(scores, 99.85)

    # در صورت همگرایی، کمی فاصله ایجاد کنیم
    if not np.isfinite(hi) or hi <= lo:
        hi = lo + max(1e-12, 1e-9 * max(1.0, np.abs(lo)))

    sev = np.zeros(n, dtype=float)
    m = scores >= lo
    sev[m] = 1.0 + (scores[m] - lo) * (98.0 - 1.0) / (hi - lo)
    sev[scores >= hi] = 99.0

    # اگر دیتاست خیلی کوچک بود و هیچ ۹۹ نشد، بزرگترین را ۹۹ کن
    if (sev == 99).sum() == 0 and n > 0:
        top_idx = int(np.argmax(scores))
        sev[top_idx] = 99.0

    return np.clip(sev, 0, 99)

def score_split(name: str, df_split: pd.DataFrame, art_split: dict):
    """اسکور با مدل/اسکیلر آموزش (سلول 8)، اما نگاشت severity بر مبنای دیتای جدید."""
    if art_split is None or df_split is None or len(df_split) == 0:
        return None, None

    scaler    = art_split["scaler"]
    if_model  = art_split["if_model"]
    feat_cols = art_split["feat_cols"]

    print(f"⚙️  [{name}] Building features ...")
    feat_df = build_features_cached(df_split)

    # تضمین وجود ستون‌ها
    for c in feat_cols:
        if c not in feat_df.columns:
            feat_df[c] = 0.0

    # فقط NON-LOCAL را اسکور کن (LOCAL ⇒ severity=0)
    mask_nonlocal = (feat_df["is_local_bucket"]==0)
    non_df = feat_df.loc[mask_nonlocal].copy()
    print(f"⚙️  [{name}] NON-LOCAL rows: {len(non_df):,}")

    if len(non_df)==0:
        # همه محلی‌اند
        feat_df["if_row_score"] = 0.0
        feat_df["if_severity"]  = 0.0
        u = feat_df.groupby("zeek.dns.query", as_index=False).agg(
            if_score=("if_row_score","max"),
            sample_ip=("source.ip","first"),
            query_length=("query_length","first"),
            dot_count=("dot_count","first"),
            digit_ratio=("digit_ratio","first"),
            entropy=("entropy","first"),
            has_hyphen=("has_hyphen","first"),
            avg_token_len=("avg_token_len","first"),
            max_token_len=("max_token_len","first"),
            vowel_ratio=("vowel_ratio","first"),
            consonant_ratio=("consonant_ratio","first"),
            nonalpha_ratio=("nonalpha_ratio","first"),
            bigram_div=("bigram_div","first"),
            tld=("tld","first"),
            is_local=("is_local","first"),
            is_guid_local=("is_guid_local","first"),
            is_ip6_ptr=("is_ip6_ptr","first"),
            is_local_bucket=("is_local_bucket","first"),
            if_severity=("if_severity","max"),
        )
        anoms = u[u["if_severity"]>0].copy()
        print(f"📊 [{name}] uniques={len(u):,} | exported=0 (all LOCAL)")
        return u, anoms

    print(f"⚙️  [{name}] Scaling + scoring IF ...")
    X = non_df[feat_cols].values
    Xs = scaler.transform(X)
    non_df["if_row_score"] = -if_model.decision_function(Xs)

    # نگاشت severity روی دیتای جدید (نه آستانه‌های آموزش)
    new_sev = map_severity_percentile_on_new(non_df["if_row_score"].values)
    non_df["if_severity"] = new_sev

    # ادغام با محلی‌ها (محلی = 0)
    feat_df = feat_df.merge(
        non_df[["zeek.dns.query","source.ip","if_row_score","if_severity"]],
        on=["zeek.dns.query","source.ip"], how="left"
    )
    feat_df["if_row_score"] = feat_df["if_row_score"].fillna(0.0)
    feat_df["if_severity"]  = feat_df["if_severity"].fillna(0.0)

    print(f"⚙️  [{name}] Aggregating to unique queries ...")
    u = (feat_df
         .groupby("zeek.dns.query", as_index=False)
         .agg(
             if_score=("if_row_score","max"),
             sample_ip=("source.ip","first"),
             query_length=("query_length","first"),
             dot_count=("dot_count","first"),
             digit_ratio=("digit_ratio","first"),
             entropy=("entropy","first"),
             has_hyphen=("has_hyphen","first"),
             avg_token_len=("avg_token_len","first"),
             max_token_len=("max_token_len","first"),
             vowel_ratio=("vowel_ratio","first"),
             consonant_ratio=("consonant_ratio","first"),
             nonalpha_ratio=("nonalpha_ratio","first"),
             bigram_div=("bigram_div","first"),
             tld=("tld","first"),
             is_local=("is_local","first"),
             is_guid_local=("is_guid_local","first"),
             is_ip6_ptr=("is_ip6_ptr","first"),
             is_local_bucket=("is_local_bucket","first"),
             if_severity=("if_severity","max"),
         ))

    anoms = u[u["if_severity"] > 0].sort_values("if_score", ascending=False).copy()
    if len(anoms) > 0:
        anoms.insert(1, "Query_Type", "IPv6" if name == "IPv6" else "IPv4")

    # گزارش درصدهای واقعی
    total_u = len(u)
    flagged = (u["if_severity"] > 0).sum()
    very_hi = (u["if_severity"] >= 99).sum()
    print(f"📊 [{name}] uniques={total_u:,} | flagged≈{flagged/total_u*100:.3f}% ({flagged}) | sev99≈{very_hi/total_u*100:.4f}% ({very_hi})")

    return u, anoms

t0 = time.time()

# ---------- Load artifacts ----------
print("🧰 Loading dual artifacts ...")
import joblib
dual = joblib.load(ARTIFACT_PATH)
art_v4 = dual.get("v4", None)
art_v6 = dual.get("v6", None)
print(f"✅ Artifacts loaded: v4={'OK' if art_v4 else 'None'} | v6={'OK' if art_v6 else 'None'}")

# ---------- Upload NEW file ----------
print("📁 Upload CSV or ZIP (with CSV) in the same raw format:")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("❌ No file uploaded.")
name, raw_bytes = next(iter(uploaded.items()))
print(f"📦 Uploaded: {name} | size={len(raw_bytes):,} bytes | md5={hashlib.md5(raw_bytes).hexdigest()}")

def _read_csv_from_zip_bytes(raw_bytes):
    zf = zipfile.ZipFile(io.BytesIO(raw_bytes))
    csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
    if not csv_names:
        raise RuntimeError("❌ No CSV found inside ZIP.")
    with zf.open(csv_names[0]) as f:
        return pd.read_csv(f, sep=None, engine="python")

if name.lower().endswith(".zip"):
    df_raw = _read_csv_from_zip_bytes(raw_bytes)
    print(f"✅ Loaded ZIP → CSV rows={len(df_raw):,}")
elif name.lower().endswith(".csv"):
    df_raw = pd.read_csv(io.BytesIO(raw_bytes), sep=None, engine="python")
    print(f"✅ Loaded CSV rows={len(df_raw):,}")
else:
    raise RuntimeError("❌ Unsupported format (CSV/ZIP only).")

# ---------- Cleanup ----------
need_cols = ["source.ip","zeek.dns.query"]
for c in need_cols:
    if c not in df_raw.columns:
        raise RuntimeError(f"❌ Missing column: {c}")

df_raw = df_raw.dropna(subset=["zeek.dns.query"]).copy()
df_raw["source.ip"] = df_raw["source.ip"].astype(str)
df_raw["zeek.dns.query"] = df_raw["zeek.dns.query"].astype(str).str.strip().str.lower()

mask_valid = df_raw["zeek.dns.query"].str.len() >= MIN_LEN
mask_valid &= ~df_raw["zeek.dns.query"].eq("-")
df_raw = df_raw.loc[mask_valid].copy()
print(f"🧹 Prefilter done. rows={len(df_raw):,}")

# ---------- Split ----------
is_v6 = df_raw["zeek.dns.query"].str.endswith("ip6.arpa", na=False)
df_v6 = df_raw[is_v6].copy()
df_v4 = df_raw[~is_v6].copy()
print(f"🔀 Split → IPv4 rows={len(df_v4):,} | IPv6 rows={len(df_v6):,}")

# ---------- Score per split ----------
u_v4, anoms_v4 = score_split("IPv4", df_v4, art_v4)
u_v6, anoms_v6 = score_split("IPv6", df_v6, art_v6)

# ---------- Save outputs ----------
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
cols_out = [
    "zeek.dns.query","Query_Type","sample_ip","if_score","if_severity",
    "query_length","dot_count","digit_ratio","entropy",
    "has_hyphen","avg_token_len","max_token_len","vowel_ratio","consonant_ratio",
    "nonalpha_ratio","bigram_div",
    "tld","is_local","is_guid_local","is_ip6_ptr","is_local_bucket"
]

merged = []
if anoms_v4 is not None and len(anoms_v4)>0:
    out_v4 = f"/content/dns_query_IF_infer_anomalies_unique_v4_{ts}.csv"
    (anoms_v4[cols_out] if set(cols_out).issubset(anoms_v4.columns) else anoms_v4).to_csv(out_v4, index=False, encoding="utf-8")
    print(f"💾 Saved IPv4 anomalies: {out_v4}")
    files.download(out_v4)
    merged.append(anoms_v4[cols_out] if set(cols_out).issubset(anoms_v4.columns) else anoms_v4)

if anoms_v6 is not None and len(anoms_v6)>0:
    out_v6 = f"/content/dns_query_IF_infer_anomalies_unique_v6_{ts}.csv"
    (anoms_v6[cols_out] if set(cols_out).issubset(anoms_v6.columns) else anoms_v6).to_csv(out_v6, index=False, encoding="utf-8")
    print(f"💾 Saved IPv6 anomalies: {out_v6}")
    files.download(out_v6)
    merged.append(anoms_v6[cols_out] if set(cols_out).issubset(anoms_v6.columns) else anoms_v6)

if merged:
    all_df = pd.concat(merged, ignore_index=True).sort_values(["Query_Type","if_score"], ascending=[True, False])
    out_all = f"/content/dns_query_IF_infer_anomalies_unique_ALL_{ts}.csv"
    all_df.to_csv(out_all, index=False, encoding="utf-8")
    print(f"💾 Saved merged anomalies: {out_all}")
    files.download(out_all)
else:
    print("ℹ️ No anomalies (Severity>0) found in either split with 0.5% policy.")

print(f"⏱ Runtime: {time.time()-t0:.2f}s")


🧰 Loading dual artifacts ...
✅ Artifacts loaded: v4=OK | v6=OK
📁 Upload CSV or ZIP (with CSV) in the same raw format:


Saving Untitled Discover session (34).zip to Untitled Discover session (34).zip
📦 Uploaded: Untitled Discover session (34).zip | size=323,672 bytes | md5=532ac25ee9320d6b0828d128838ca96f
✅ Loaded ZIP → CSV rows=112,896
🧹 Prefilter done. rows=108,698
🔀 Split → IPv4 rows=108,228 | IPv6 rows=470
⚙️  [IPv4] Building features ...
🔧 Building cached features on 1,311 unique queries ...
⚙️  [IPv4] NON-LOCAL rows: 53,109
⚙️  [IPv4] Scaling + scoring IF ...
⚙️  [IPv4] Aggregating to unique queries ...
📊 [IPv4] uniques=1,311 | flagged≈6.484% (85) | sev99≈1.0679% (14)
⚙️  [IPv6] Building features ...
🔧 Building cached features on 2 unique queries ...
⚙️  [IPv6] NON-LOCAL rows: 470
⚙️  [IPv6] Scaling + scoring IF ...
⚙️  [IPv6] Aggregating to unique queries ...
📊 [IPv6] uniques=2 | flagged≈50.000% (1) | sev99≈50.0000% (1)
💾 Saved IPv4 anomalies: /content/dns_query_IF_infer_anomalies_unique_v4_20251028_060620.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

💾 Saved IPv6 anomalies: /content/dns_query_IF_infer_anomalies_unique_v6_20251028_060620.csv


...

<IPython.core.display.Javascript object>

💾 Saved merged anomalies: /content/dns_query_IF_infer_anomalies_unique_ALL_20251028_060620.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⏱ Runtime: 23.76s


In [11]:
# =======================
# 🔹 Cell 10 — 60-minute IP×Query counts (no modeling)
#     - Input: raw df from Cell 1 (same columns)
#     - Output columns (in order): Source IP, Query, Window Start, Count
# =======================
import numpy as np
import pandas as pd
from google.colab import files

# --- 0) Sanity: expect raw df from Cell 1
if "df" not in globals():
    raise RuntimeError("❌ DataFrame `df` not found. لطفاً سلول 1 را اجرا کنید تا df ساخته شود.")

df_raw = df.copy()

# --- 1) Minimal cleanup (optional) ---
for col in ["@timestamp", "source.ip", "zeek.dns.query"]:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].replace("-", np.nan)

# --- 2) Fast timestamp parsing (2-pass) ---
if "@timestamp" not in df_raw.columns:
    raise RuntimeError("❌ ستون '@timestamp' وجود ندارد.")

ts_str = df_raw["@timestamp"].astype(str).str.replace(" @ ", " ", regex=False)
ts1 = pd.to_datetime(ts_str, format="%b %d, %Y %H:%M:%S.%f", errors="coerce")
mask = ts1.isna()
if mask.any():
    ts2 = pd.to_datetime(ts_str[mask], format="%b %d, %Y %H:%M:%S", errors="coerce")
    ts1.loc[mask] = ts2

df_raw["ts"] = ts1

# --- 3) Drop rows with missing essentials ---
need = ["source.ip","zeek.dns.query","ts"]
missing_cols = [c for c in need if c not in df_raw.columns]
if missing_cols:
    raise RuntimeError(f"❌ ستون‌های لازم موجود نیست: {missing_cols}")

df_raw = df_raw.dropna(subset=need)
df_raw["source.ip"] = df_raw["source.ip"].astype(str).str.strip()
df_raw["zeek.dns.query"] = df_raw["zeek.dns.query"].astype(str).str.strip()

# --- 4) Windowing (60 minutes) ---
df_raw["window_60m"] = df_raw["ts"].dt.floor("60min")

# --- 5) Group & count per (source.ip, zeek.dns.query, window_60m) ---
grp = (df_raw
       .groupby(["source.ip","window_60m","zeek.dns.query"])
       .size()
       .reset_index(name="Count"))

# --- 6) Reorder & rename columns ---
out = grp.rename(columns={
    "source.ip": "Source IP",
    "zeek.dns.query": "Query",
    "window_60m": "Window Start"
})[["Source IP", "Query", "Window Start", "Count"]]

# مرتب‌سازی
out = out.sort_values(["Window Start","Source IP","Count"], ascending=[True,True,False]).reset_index(drop=True)

# فرمت تاریخ برای خوانایی
try:
    out["Window Start"] = pd.to_datetime(out["Window Start"]).dt.strftime("%Y-%m-%d %H:%M:%S")
except Exception:
    pass

print(f"✅ Built 60-minute IP×Query counts: {len(out):,} rows")
display(out.head(10))

# --- 7) Save & download ---
save_path = "/content/ip_query_counts_60min.csv"
out.to_csv(save_path, index=False, encoding="utf-8")
print(f"💾 Saved: {save_path}")
files.download(save_path)


✅ Built 60-minute IP×Query counts: 75,251 rows


,Source IP,Query,Window Start,Count
0,10.10.10.188,_rdlink._tcp.local,2025-09-17 10:00:00,24
1,10.10.10.188,time.apple.com,2025-09-17 10:00:00,2
2,10.10.10.200,_sleep-proxy._udp.local,2025-09-17 10:00:00,48
3,10.10.10.200,hamids-iphone.local,2025-09-17 10:00:00,36
4,10.10.10.200,528705ba._sub._apple-mobdev2._tcp.local,2025-09-17 10:00:00,12
5,10.10.10.200,gateway.icloud.com,2025-09-17 10:00:00,4
6,10.10.10.200,chat.cdn.whatsapp.net,2025-09-17 10:00:00,2
7,10.10.10.200,gateway.fe2.apple-dns.net,2025-09-17 10:00:00,2
8,10.10.10.200,get-bx.g.aaplimg.com,2025-09-17 10:00:00,2
9,10.10.10.200,gs-loc.ls-apple.com.akadns.net,2025-09-17 10:00:00,2


💾 Saved: /content/ip_query_counts_60min.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
# =======================
# 🔹 Cell 11 — Feature Engineering (5m window) with stable Top/HiEntropy in df_feat
# =======================
import numpy as np
import pandas as pd
import re, time
from collections import Counter

t0 = time.time()

# ---------- Config ----------
WINDOW = "5min"      # اگر 60 دقیقه می‌خواهی: "60min"
BIN_FOR_SPIKE = "10s"
MIN_ROWS_FOR_IAT = 3

# ---------- 0) Expect raw df from Cell 1 ----------
if 'df' not in globals():
    raise RuntimeError("`df` (raw Zeek DNS DataFrame) not found. Run Cell 1 first.")

df_raw = df.copy()

# ---------- 1) Minimal cleaning ----------
for col in ["zeek.dns.TTLs","zeek.dns.answers","zeek.dns.rtt","zeek.dns.rejected","zeek.dns.TC"]:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].replace("-", np.nan)

# ---------- 2) Fast timestamp parse ----------
ts_str = df_raw["@timestamp"].astype(str).str.replace(" @ ", " ", regex=False)
ts1 = pd.to_datetime(ts_str, format="%b %d, %Y %H:%M:%S.%f", errors="coerce")
mask = ts1.isna()
if mask.any():
    ts2 = pd.to_datetime(ts_str[mask], format="%b %d, %Y %H:%M:%S", errors="coerce")
    ts1.loc[mask] = ts2
df_raw["ts"] = ts1
df_raw = df_raw.dropna(subset=["source.ip","zeek.dns.query","ts"]).copy()

# ---------- 3) Casts ----------
df_raw["zeek.dns.rtt"] = pd.to_numeric(df_raw.get("zeek.dns.rtt"), errors="coerce")
for col in ["zeek.dns.rejected","zeek.dns.TC"]:
    if col in df_raw.columns:
        s = df_raw[col].astype(str).str.lower()
        df_raw[col] = np.where(s.eq("true"), 1.0, 0.0)

# ---------- 4) Row-wise lexical features ----------
q_series = df_raw["zeek.dns.query"].astype(str)

def entropy_str_fast(s: str):
    if not s: return 0.0
    cnt = np.fromiter(Counter(s).values(), dtype=float)
    p = cnt / cnt.sum()
    return float(-(p * np.log2(p)).sum())

uq = pd.unique(q_series)
ent_map  = {s: entropy_str_fast(s) for s in uq}
df_raw["domain_length"]   = q_series.str.len()
df_raw["subdomain_depth"] = q_series.str.count(r"\.")
df_raw["entropy"]         = q_series.map(ent_map)

# ---------- 5) TTL & Answers parsing ----------
def _parse_ttls_safe(ttl_str):
    if not isinstance(ttl_str, str):
        return []
    vals = []
    for tok in ttl_str.split(","):
        tok = tok.strip()
        if not tok: continue
        tok = tok.replace(",", "")
        try:
            vals.append(float(tok))
        except:
            nums = re.findall(r"\d+", tok)
            if nums:
                try: vals.append(float("".join(nums)))
                except: pass
    return vals

df_raw["ttl_list"] = df_raw.get("zeek.dns.TTLs", "").fillna("").astype(str).map(_parse_ttls_safe)
df_raw["ttl_mean_row"] = df_raw["ttl_list"].map(lambda L: float(np.mean(L)) if L else np.nan)
df_raw["ttl_std_row"]  = df_raw["ttl_list"].map(lambda L: float(np.std (L)) if L else np.nan)

def _answer_tokens(s):
    if not isinstance(s, str) or not s.strip(): return []
    return [tok.strip() for tok in s.split(",") if tok.strip()]

df_raw["_ans_tokens"] = df_raw.get("zeek.dns.answers","").fillna("").astype(str).map(_answer_tokens)

# ساده: شمارش IP در پاسخ‌ها
re_ip4 = re.compile(r"\b((?:\d{1,3}\.){3}\d{1,3})\b")
re_ip6 = re.compile(r"\b([0-9a-fA-F:]{2,})\b")

def extract_ips_from_tokens(tokens):
    ips4, ips6 = [], []
    for t in tokens:
        m4 = re_ip4.findall(t)
        if m4: ips4.extend(m4)
        else:
            m6 = re_ip6.findall(t)
            for cand in m6:
                if ":" in cand:
                    ips6.append(cand)
    return list(set(ips4)), list(set(ips6))

df_raw["_ans_ip4"], df_raw["_ans_ip6"] = zip(*df_raw["_ans_tokens"].map(extract_ips_from_tokens))

# ---------- 6) Transport ----------
transport = df_raw.get("network.transport")
df_raw["is_tcp_row"] = (transport.astype(str).str.lower().eq("tcp")).astype(float) if transport is not None else 0.0

# ---------- 7) Windowing ----------
time_col = f"window_{WINDOW}"
df_raw[time_col] = df_raw["ts"].dt.floor(WINDOW)
group_keys = ["source.ip", time_col]

# ---------- 8) Base aggregations ----------
agg = df_raw.groupby(group_keys).agg(
    dns_count             = ("zeek.dns.query","count"),
    unique_domains        = ("zeek.dns.query", pd.Series.nunique),
    avg_domain_len        = ("domain_length","mean"),
    avg_entropy           = ("entropy","mean"),
    avg_subdomain_depth   = ("subdomain_depth","mean"),
    avg_rtt               = ("zeek.dns.rtt","mean"),
    std_rtt               = ("zeek.dns.rtt","std"),
    avg_ttl               = ("ttl_mean_row","mean"),
    std_ttl               = ("ttl_std_row","mean"),
    avg_answer_count      = ("_ans_tokens", lambda x: float(np.mean([len(t) for t in x]) if len(x)>0 else 0.0)),
    tcp_ratio             = ("is_tcp_row","mean"),
).reset_index()

# ---------- 9) Resolver diversity ----------
resolvers = df_raw.groupby(group_keys)["destination.ip"].nunique().reset_index(name="resolver_unique")
agg = agg.merge(resolvers, on=group_keys, how="left")

# ---------- 10) QTYPE ratios ----------
tmp_q = (df_raw.groupby(group_keys + ["zeek.dns.qtype_name"]).size()
         .reset_index(name="cnt"))
pv_q = tmp_q.pivot_table(index=group_keys, columns="zeek.dns.qtype_name", values="cnt", fill_value=0).reset_index()
pv_q.columns = [c if c in group_keys else f"qtype_cnt_{c}" for c in pv_q.columns]
pv_q["qtype_total"] = pv_q.drop(columns=group_keys).sum(axis=1)
for qn in ["A","AAAA","TXT","MX","NS","CNAME","SRV","ANY"]:
    col = f"qtype_cnt_{qn}"
    if col not in pv_q.columns:
        pv_q[col] = 0
    pv_q[f"qtype_ratio_{qn}"] = np.where(pv_q["qtype_total"]>0, pv_q[col]/pv_q["qtype_total"], 0.0)
qtype_cols = [c for c in pv_q.columns if c.startswith("qtype_ratio_")]
agg = agg.merge(pv_q[group_keys + qtype_cols], on=group_keys, how="left")

# ---------- 11) RCODE ratios ----------
tmp_r = (df_raw.groupby(group_keys + ["zeek.dns.rcode_name"]).size()
         .reset_index(name="rcnt"))
pv_r = tmp_r.pivot_table(index=group_keys, columns="zeek.dns.rcode_name", values="rcnt", fill_value=0).reset_index()
pv_r.columns = [c if c in group_keys else f"rcode_cnt_{c}" for c in pv_r.columns]
for rn in ["NXDOMAIN","SERVFAIL","REFUSED","NOERROR"]:
    if f"rcode_cnt_{rn}" not in pv_r.columns:
        pv_r[f"rcode_cnt_{rn}"] = 0
pv_r["rcode_total"] = pv_r[[c for c in pv_r.columns if c.startswith("rcode_cnt_")]].sum(axis=1)
for rn in ["NXDOMAIN","SERVFAIL","REFUSED"]:
    pv_r[f"{rn.lower()}_ratio"] = np.where(pv_r["rcode_total"]>0, pv_r[f"rcode_cnt_{rn}"]/pv_r["rcode_total"], 0.0)
rcode_cols = ["nxdomain_ratio","servfail_ratio","refused_ratio"]
agg = agg.merge(pv_r[group_keys + rcode_cols], on=group_keys, how="left")

# ---------- 12) Fast-flux proxies ----------
ans_ip_counts = (
    df_raw.assign(ans_ip4_count=df_raw["_ans_ip4"].map(len),
                  ans_ip6_count=df_raw["_ans_ip6"].map(len))
          .groupby(group_keys)[["ans_ip4_count","ans_ip6_count"]]
          .sum().reset_index()
)
agg = agg.merge(ans_ip_counts, on=group_keys, how="left")
agg["answer_ip_unique_total"] = agg["ans_ip4_count"].fillna(0) + agg["ans_ip6_count"].fillna(0)
agg["ttl_cv"] = np.where(agg["avg_ttl"].fillna(0) > 0,
                         agg["std_ttl"].fillna(0) / agg["avg_ttl"].replace(0, np.nan),
                         0.0)

# ---------- 13) NXDOMAIN→NOERROR within window ----------
q_rc = (df_raw.groupby(group_keys + ["zeek.dns.query"])["zeek.dns.rcode_name"]
        .apply(lambda s: set(s.dropna().tolist()))
        .reset_index(name="rcode_set"))
q_rc["has_nx_then_ok"] = q_rc["rcode_set"].apply(lambda st: int("NXDOMAIN" in st and "NOERROR" in st))
nx_ok = q_rc.groupby(group_keys)["has_nx_then_ok"].sum().reset_index(name="nx_then_ok_domains")
agg = agg.merge(nx_ok, on=group_keys, how="left")
agg["nx_then_ok_ratio"] = np.where(agg["unique_domains"]>0,
                                   agg["nx_then_ok_domains"].fillna(0)/agg["unique_domains"],
                                   0.0)

# ---------- 14) Periodicity / Beaconing proxies ----------
def iat_stats_on_group(times: pd.Series) -> pd.Series:
    times = pd.to_datetime(times).sort_values().values
    if len(times) < MIN_ROWS_FOR_IAT:
        return pd.Series({"iat_median": np.nan, "iat_mean": np.nan, "iat_std": np.nan, "iat_cv": np.nan})
    diffs = np.diff(times).astype("timedelta64[ms]").astype(float) / 1000.0  # seconds
    mu  = float(np.mean(diffs))
    sd  = float(np.std(diffs))
    med = float(np.median(diffs))
    cv  = float(sd / mu) if mu > 0 else np.nan
    return pd.Series({"iat_median": med, "iat_mean": mu, "iat_std": sd, "iat_cv": cv})

iat_df = (df_raw.groupby(group_keys).apply(lambda g: iat_stats_on_group(g["ts"])).reset_index())
agg = agg.merge(iat_df, on=group_keys, how="left")

agg["periodic_flag"] = ((agg["dns_count"] >= 8) & (agg["iat_cv"].fillna(1e9) < 0.10)).astype(int)

# ---------- 15) Spikes (10s bins) ----------
df_bins = df_raw.copy()
df_bins["bin_ts"] = df_bins["ts"].dt.floor(BIN_FOR_SPIKE)
bin_ct = (df_bins.groupby(group_keys + ["bin_ts"]).size().reset_index(name="bin_cnt"))
peak = bin_ct.groupby(group_keys)["bin_cnt"].max().reset_index(name="peak_10s")
avg_bin = bin_ct.groupby(group_keys)["bin_cnt"].mean().reset_index(name="avg_bin_10s")
agg = agg.merge(peak, on=group_keys, how="left").merge(avg_bin, on=group_keys, how="left")
agg["spike_ratio"] = np.where(agg["avg_bin_10s"].fillna(0)>0,
                              agg["peak_10s"].fillna(0)/agg["avg_bin_10s"],
                              0.0)

# ---------- 16) Derived simple rates ----------
win_minutes = (pd.Timedelta(WINDOW).total_seconds() / 60.0)
win_seconds = pd.Timedelta(WINDOW).total_seconds()
agg["qpm"] = agg["dns_count"] / max(win_minutes, 1.0)
agg["qps"] = agg["dns_count"] / max(win_seconds, 1.0)
agg["unique_domain_ratio"] = np.where(agg["dns_count"]>0, agg["unique_domains"]/agg["dns_count"], 0.0)
agg["error_ratio"] = (agg["nxdomain_ratio"].fillna(0) + agg["servfail_ratio"].fillna(0) + agg["refused_ratio"].fillna(0)).clip(0,1)
agg["udp_ratio"] = 1.0 - agg["tcp_ratio"].fillna(0.0)

# ---------- 17) Top & Highest-Entropy Domain (stable, inside df_feat) ----------
# شمارش کوئری‌ها
vc_d = (df_raw.groupby(group_keys + ["zeek.dns.query"]).size().reset_index(name="cnt"))
# TopDomain
top1d = (vc_d.sort_values(group_keys + ["cnt"], ascending=[True, True, False])
              .groupby(group_keys).head(1)
              .rename(columns={"zeek.dns.query":"TopDomain","cnt":"TopDomainCount"}))
# Highest Entropy Domain (از entropy ردیفی خود df_raw)
df_q_stats = (df_raw.groupby(group_keys + ["zeek.dns.query"])
              .agg(entropy_mean=("entropy","mean"), qlen=("zeek.dns.query", lambda s: len(str(s.iloc[0]))))
              .reset_index())
hiE = (df_q_stats.sort_values(group_keys + ["entropy_mean","qlen"], ascending=[True, True, False, False])
                 .groupby(group_keys).head(1)
                 .rename(columns={"zeek.dns.query":"HiEntropyDomain",
                                  "entropy_mean":"HiEntropyValue"}))

agg = (agg.merge(top1d, on=group_keys, how="left")
          .merge(hiE,   on=group_keys, how="left"))

# ---------- 18) Final cleaning ----------
num_cols = [c for c in agg.columns if c not in ["source.ip", time_col, "TopDomain","HiEntropyDomain"]]
for c in num_cols:
    if pd.api.types.is_numeric_dtype(agg[c]):
        agg[c] = agg[c].astype(float).fillna(0.0)

# اگر Top/HiEntropy خالی بود، با رشته‌ی خالی پُر کن
for c in ["TopDomain","HiEntropyDomain"]:
    if c not in agg.columns:
        agg[c] = ""
    agg[c] = agg[c].fillna("").astype(str).str.strip().str.lower()
if "TopDomainCount" not in agg.columns:
    agg["TopDomainCount"] = 0.0
if "HiEntropyValue" not in agg.columns:
    agg["HiEntropyValue"] = 0.0

feat_cols = [
    "dns_count","unique_domains","unique_domain_ratio","qpm","qps",
    "nxdomain_ratio","servfail_ratio","refused_ratio","error_ratio","nx_then_ok_ratio",
    "avg_domain_len","avg_entropy","avg_subdomain_depth",
    "avg_rtt","std_rtt","iat_median","iat_mean","iat_std","iat_cv","spike_ratio",
    "avg_ttl","std_ttl","ttl_cv","avg_answer_count","answer_ip_unique_total",
    "tcp_ratio","udp_ratio","resolver_unique",
    "qtype_ratio_A","qtype_ratio_AAAA","qtype_ratio_TXT","qtype_ratio_MX","qtype_ratio_NS","qtype_ratio_CNAME","qtype_ratio_SRV","qtype_ratio_ANY",
    "periodic_flag",
]
for c in feat_cols:
    if c not in agg.columns:
        agg[c] = 0.0

col_order = ["source.ip", time_col, "TopDomain","TopDomainCount","HiEntropyDomain","HiEntropyValue"] + feat_cols
df_feat = agg[col_order].copy()

print(f"✅ Feature windows built: {len(df_feat):,} rows | window={WINDOW}")
print(f"⏱ Feature engineering time: {time.time()-t0:.2f}s")
display(df_feat.head(10))


/tmp/ipython-input-2390224066.py:197: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  iat_df = (df_raw.groupby(group_keys).apply(lambda g: iat_stats_on_group(g["ts"])).reset_index())


✅ Feature windows built: 22,208 rows | window=5min
⏱ Feature engineering time: 99.95s


,source.ip,window_5min,TopDomain,TopDomainCount,HiEntropyDomain,HiEntropyValue,dns_count,unique_domains,unique_domain_ratio,qpm,...,resolver_unique,qtype_ratio_A,qtype_ratio_AAAA,qtype_ratio_TXT,qtype_ratio_MX,qtype_ratio_NS,qtype_ratio_CNAME,qtype_ratio_SRV,qtype_ratio_ANY,periodic_flag
0,10.10.10.18,2025-09-17 13:25:00,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",30.0,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",4.457519,185.0,31.0,0.167568,37.0,...,3.0,0.324324,0.043243,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,10.10.10.18,2025-09-17 13:30:00,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",45.0,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",4.457519,130.0,6.0,0.046154,26.0,...,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,10.10.10.18,2025-09-17 13:35:00,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",60.0,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",4.457519,95.0,5.0,0.052632,19.0,...,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,10.10.10.18,2025-09-17 13:40:00,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",55.0,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",4.457519,100.0,3.0,0.030000,20.0,...,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,10.10.10.18,2025-09-17 13:45:00,android.local,20.0,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",4.457519,45.0,4.0,0.088889,9.0,...,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,10.10.10.18,2025-09-17 13:50:00,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",10.0,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",4.457519,10.0,1.0,0.100000,2.0,...,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,10.10.10.18,2025-09-17 14:05:00,_mi-connect._udp.local,15.0,_mi-connect._udp.local,3.788755,15.0,1.0,0.066667,3.0,...,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,10.10.10.18,2025-09-17 14:10:00,_mi-connect._udp.local,15.0,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",4.457519,40.0,3.0,0.075000,8.0,...,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,10.10.10.18,2025-09-17 14:15:00,18.10.10.10.in-addr.arpa,15.0,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",4.457519,50.0,4.0,0.080000,10.0,...,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,10.10.10.18,2025-09-17 14:20:00,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",30.0,"{""nm"":""arsham"",""as"":""[8194]"",""ip"":""18""}._mi-co...",4.457519,30.0,1.0,0.033333,6.0,...,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
# =======================
# 🔹 Cell 12 — Train Bagged IsolationForest on windowed features (df_feat)
#     - Focus on TopDomain / HiEntropyDomain (with higher weight)
#     - Local-like domains never flagged
#     - Severity policy: 6% >0, 0.3% = 99
#     - Robust time formatting
# =======================
import time, os, re
import numpy as np
import pandas as pd
from collections import Counter
from numpy.random import default_rng
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
import joblib

t0 = time.time()

ARTIFACT_PATH  = "/content/if_window_artifacts.joblib"
OUT_ALERTS_CSV = "/content/anomaly_windows_if_alerts.csv"

# --- Severity policy (requested) ---
TAIL_PCT    = 94.0   # top 6% → >0
EXTREME_PCT = 99.7   # top 0.3% → 99

if 'df_feat' not in globals():
    raise RuntimeError("`df_feat` not found. Run Cell 11 first.")

# detect time window column
time_col_candidates = [c for c in df_feat.columns if isinstance(c, str) and c.startswith("window_")]
if not time_col_candidates:
    raise RuntimeError("No time window column found in df_feat (expected 'window_5min' or 'window_60min').")
time_col = time_col_candidates[0]

# ensure columns exist
for c in ["TopDomain","TopDomainCount","HiEntropyDomain","HiEntropyValue"]:
    if c not in df_feat.columns:
        df_feat[c] = "" if "Domain" in c else 0.0
df_feat["TopDomain"]       = df_feat["TopDomain"].fillna("").astype(str).str.strip().str.lower()
df_feat["HiEntropyDomain"] = df_feat["HiEntropyDomain"].fillna("").astype(str).str.strip().str.lower()
df_feat["TopDomainCount"]  = pd.to_numeric(df_feat["TopDomainCount"], errors="coerce").fillna(0.0)
df_feat["HiEntropyValue"]  = pd.to_numeric(df_feat["HiEntropyValue"], errors="coerce").fillna(0.0)

# --- lexical helpers for local-like
RE_GUID_LOCAL  = re.compile(r'^[0-9a-f]{8}-(?:[0-9a-f]{4}-){3}[0-9a-f]{12}\.local$', re.IGNORECASE)
def is_local_like(domain: str) -> int:
    if not domain: return 0
    d = domain.strip().lower()
    if RE_GUID_LOCAL.match(d): return 1
    if d.endswith(".local"):  return 1
    if "." not in d:          return 1
    return 0

def entropy_str(s: str) -> float:
    if not isinstance(s, str) or s == "":
        return 0.0
    cnt = np.fromiter(Counter(s).values(), dtype=float)
    p = cnt / cnt.sum()
    return float(-(p * np.log2(p)).sum())

# build lexical maps for both domains
for col in ["TopDomain","HiEntropyDomain"]:
    df_feat[col] = df_feat[col].fillna("").astype(str).str.strip().str.lower()

uq_top = df_feat["TopDomain"].unique()
uq_hie = df_feat["HiEntropyDomain"].unique()

top_ent = {s: entropy_str(s) for s in uq_top}
top_len = {s: len(s) for s in uq_top}
top_dig = {s: (sum(c.isdigit() for c in s)/len(s) if len(s)>0 else 0.0) for s in uq_top}
top_vow = {s: (sum(ch in "aeiou" for ch in s)/len(s) if len(s)>0 else 0.0) for s in uq_top}
top_loc = {s: is_local_like(s) for s in uq_top}

hie_ent = {s: entropy_str(s) for s in uq_hie}
hie_len = {s: len(s) for s in uq_hie}
hie_dig = {s: (sum(c.isdigit() for c in s)/len(s) if len(s)>0 else 0.0) for s in uq_hie}
hie_vow = {s: (sum(ch in "aeiou" for ch in s)/len(s) if len(s)>0 else 0.0) for s in uq_hie}
hie_loc = {s: is_local_like(s) for s in uq_hie}

df_work = df_feat.copy()

df_work["top1_entropy"]     = df_work["TopDomain"].map(top_ent)
df_work["top1_length"]      = df_work["TopDomain"].map(top_len)
df_work["top1_digit_ratio"] = df_work["TopDomain"].map(top_dig)
df_work["top1_vowel_ratio"] = df_work["TopDomain"].map(top_vow)
df_work["top1_is_local"]    = df_work["TopDomain"].map(top_loc).astype(int)

# HiEntropyValue را اگر آماده داریم، همان را استفاده می‌کنیم؛ وگرنه محاسبه از اسم دامنه
df_work["hiE_entropy"]      = np.where(df_work["HiEntropyValue"].notna() & (df_work["HiEntropyValue"]>0),
                                       df_work["HiEntropyValue"], df_work["HiEntropyDomain"].map(hie_ent))
df_work["hiE_length"]       = df_work["HiEntropyDomain"].map(hie_len)
df_work["hiE_digit_ratio"]  = df_work["HiEntropyDomain"].map(hie_dig)
df_work["hiE_vowel_ratio"]  = df_work["HiEntropyDomain"].map(hie_vow)
df_work["hiE_is_local"]     = df_work["HiEntropyDomain"].map(hie_loc).astype(int)

# top1 سهم
if "top1_share" not in df_work.columns:
    if "dns_count" in df_work.columns:
        df_work["top1_share"] = np.where(
            df_work["dns_count"]>0,
            df_work["TopDomainCount"].fillna(0)/df_work["dns_count"],
            0.0
        )
    else:
        df_work["top1_share"] = 0.0

# --- feature set with higher weight on entropy signals ---
base_focus = [
    "top1_entropy","top1_length","top1_digit_ratio","top1_vowel_ratio","top1_share",
    "hiE_entropy","hiE_length","hiE_digit_ratio","hiE_vowel_ratio",
    "avg_entropy","unique_domain_ratio","qpm","qps",
    "nxdomain_ratio","servfail_ratio","refused_ratio","error_ratio",
]
for c in base_focus:
    if c not in df_work.columns:
        df_work[c] = 0.0

weighted_feats = base_focus + ["top1_entropy","hiE_entropy","top1_entropy","hiE_entropy"]  # وزن بالاتر

# local-like windows must never be anomalies
local_mask = (df_work["top1_is_local"]==1) | (df_work["hiE_is_local"]==1)

# ---------- Train IF (bagged) ----------
scaler = RobustScaler()
X = scaler.fit_transform(df_work[weighted_feats].values)
print(f"ℹ️ Training dataset (focused) shape: {X.shape}")

rng = default_rng(42)
idx_all = np.arange(len(weighted_feats))
subA = rng.choice(idx_all, size=max(1,int(len(idx_all)*0.65)), replace=False)
subB = rng.choice(idx_all, size=max(1,int(len(idx_all)*0.65)), replace=False)

def choose_max_samples(n_rows, frac=0.60, cap=60000):
    return int(min(max(2048, int(n_rows*frac)), cap))

n_rows = X.shape[0]
specs = [
    ("IF_full", idx_all, 400, choose_max_samples(n_rows, 0.60), 0.005, 17),
    ("IF_subA", subA,     350, choose_max_samples(n_rows, 0.55), 0.007, 29),
    ("IF_subB", subB,     350, choose_max_samples(n_rows, 0.55), 0.007, 71),
]

scores_list, models = [], []
for name, cols_idx, n_estimators, max_samples, contamination, seed in specs:
    t1 = time.time()
    ifm = IsolationForest(
        n_estimators=n_estimators,
        contamination=contamination,
        max_samples=max_samples,
        random_state=seed,
        n_jobs=-1,
        bootstrap=False
    )
    ifm.fit(X[:, cols_idx])
    s = -ifm.decision_function(X[:, cols_idx])
    scores_list.append(s)
    models.append({"name": name, "cols_idx": cols_idx, "model": ifm})
    print(f"🧩 {name} | trees={n_estimators} | max_samples={max_samples} | fit+score: {time.time()-t1:.2f}s | mean={s.mean():.4f}")

scores_mat = np.vstack(scores_list).T
combined_if = scores_mat.max(axis=1)

# ---------- Severity mapping (6% / 0.3%) ----------
def sev_from(vals, tail=TAIL_PCT, extreme=EXTREME_PCT):
    p_tail = np.percentile(vals, tail)
    p_ext  = np.percentile(vals, extreme)
    lo, hi = p_tail, max(p_ext, p_tail + 1e-12)
    sev = np.zeros_like(vals, dtype=float)
    m = vals >= lo
    sev[m] = 1.0 + (vals[m] - lo) * (98.0 - 1.0) / (hi - lo)
    sev[vals >= hi] = 99.0
    return np.clip(sev, 0, 99)

sev_if   = sev_from(combined_if, TAIL_PCT, EXTREME_PCT)
sev_top1 = sev_from(df_work["top1_entropy"].values, TAIL_PCT, EXTREME_PCT)
sev_hiE  = sev_from(df_work["hiE_entropy"].values,  TAIL_PCT, EXTREME_PCT)

final_sev = np.maximum(sev_if, np.maximum(sev_top1, sev_hiE))
final_sev[local_mask.values] = 0.0

df_work["if_score_combined"] = combined_if
df_work["if_severity"] = final_sev
df_work["if_label"] = (final_sev > 0).astype(int)

print(f"\n📊 Rows: {len(df_work):,} | Sev>0: {(final_sev>0).sum()} ({(final_sev>0).mean()*100:.3f}%) | Sev=99: {(final_sev>=99).sum()} ({(final_sev>=99).mean()*100:.4f}%)")
print(f"⏱ Train time: {time.time()-t0:.2f}s")

# ---------- save artifacts ----------
artifacts = {
    "scaler": scaler,
    "models": models,
    "feat_cols": weighted_feats,
    "policy": {"tail_pct": float(TAIL_PCT), "extreme_pct": float(EXTREME_PCT)},
    "notes":  "Dual-domain focus from df_feat (TopDomain/HiEntropyDomain). Local-like windows forced to severity=0."
}
try:
    joblib.dump(artifacts, ARTIFACT_PATH)
    print(f"🧰 Artifacts saved → {ARTIFACT_PATH}")
except Exception as e:
    print(f"⚠️ Failed to save artifacts: {e}")

# ---------- export (human-readable) ----------
def robust_parse_series(srs):
    if np.issubdtype(srs.dtype, np.datetime64):
        return pd.to_datetime(srs, errors="coerce")
    s = srs.astype(str).fillna("")
    parsed = pd.to_datetime(s, errors="coerce")
    try:
        parsed = parsed.dt.tz_localize(None)
    except Exception:
        pass
    return parsed

def format_dt_safe(ts):
    if pd.isna(ts): return ""
    try:
        y,m,d = ts.year, ts.month, ts.day
        H,M,S = ts.hour, ts.minute, ts.second
        if y < 1970 or y > 2100:
            return ""
        return f"{y:04d}-{m:02d}-{d:02d} {H:02d}:{M:02d}:{S:02d}"
    except Exception:
        return ""

df_out = df_work.copy()
parsed = robust_parse_series(df_out[time_col]) if time_col in df_out.columns else pd.Series([pd.NaT]*len(df_out))
df_out["Window Start"] = [format_dt_safe(t) for t in parsed]

if "source.ip" in df_out.columns:
    df_out.rename(columns={"source.ip":"Source IP"}, inplace=True)
elif "Source IP" not in df_out.columns:
    df_out["Source IP"] = ""

out_cols = [
    "Source IP","Window Start",
    "TopDomain","TopDomainCount","HiEntropyDomain","HiEntropyValue",
    "top1_entropy","hiE_entropy","top1_length","hiE_length",
    "top1_digit_ratio","hiE_digit_ratio","top1_vowel_ratio","hiE_vowel_ratio",
    "top1_share","avg_entropy","unique_domain_ratio","qpm","qps",
    "nxdomain_ratio","servfail_ratio","refused_ratio","error_ratio",
    "if_score_combined","if_severity","if_label"
]
out_cols = [c for c in out_cols if c in df_out.columns]

alerts = df_out[df_out["if_severity"] > 0].sort_values(
    ["if_severity","hiE_entropy","top1_entropy","qps"], ascending=[False, False, False, False]
)
alerts[out_cols].to_csv(OUT_ALERTS_CSV, index=False, encoding="utf-8")
print(f"💾 Saved: {OUT_ALERTS_CSV} | Alerts: {len(alerts):,}")

try:
    from google.colab import files
    files.download(OUT_ALERTS_CSV)
except Exception:
    pass

print(f"⏱ Cell 12 runtime: {time.time()-t0:.2f}s")


ℹ️ Training dataset (focused) shape: (22208, 21)
🧩 IF_full | trees=400 | max_samples=13324 | fit+score: 2.22s | mean=-0.2025
🧩 IF_subA | trees=350 | max_samples=12214 | fit+score: 1.87s | mean=-0.1883
🧩 IF_subB | trees=350 | max_samples=12214 | fit+score: 2.04s | mean=-0.1763

📊 Rows: 22,208 | Sev>0: 814 (3.665%) | Sev=99: 14 (0.0630%)
⏱ Train time: 6.27s
🧰 Artifacts saved → /content/if_window_artifacts.joblib
💾 Saved: /content/anomaly_windows_if_alerts.csv | Alerts: 814


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⏱ Cell 12 runtime: 7.05s


In [15]:
# 📄 نمایش فایل خروجی anomaly_windows_if_alerts.csv در Colab

import pandas as pd

# مسیر فایل خروجی سلول 12
csv_path = "/content/anomaly_windows_if_alerts.csv"

# خواندن و نمایش داده‌ها
df_alerts = pd.read_csv(csv_path)

# فقط چند ردیف اول برای چک سریع
print(f"📊 تعداد کل رکوردهای هشدار: {len(df_alerts):,}")
display(df_alerts.head(50))  # ۵۰ سطر اول برای بررسی سریع

# اگر خواستی کل جدول را در خروجی HTML با اسکرول ببینی، از این استفاده کن:
from IPython.display import HTML
HTML(df_alerts.to_html(max_rows=100, index=False))


📊 تعداد کل رکوردهای هشدار: 814


,Source IP,Window Start,TopDomain,TopDomainCount,HiEntropyDomain,HiEntropyValue,top1_entropy,hiE_entropy,top1_length,hiE_length,...,unique_domain_ratio,qpm,qps,nxdomain_ratio,servfail_ratio,refused_ratio,error_ratio,if_score_combined,if_severity,if_label
0,10.10.10.69,2025-09-20 13:05:00,content-autofill.googleapis.com,30.0,xga4p6accpq7k2gopuvq-f-6cb4e5303-clientnsv4-s....,4.709202,3.821312,4.709202,31,58,...,0.183158,95.0,1.583333,0.000000,0.008421,0.000000,0.008421,0.012037,99.000000,1
1,192.168.100.5,2025-10-07 11:45:00,a1968.i6g1.akamai.net,24.0,activationedge-fabrick-1124129460.us-west-2.el...,4.573459,3.499228,4.573459,21,61,...,0.092391,368.0,6.133333,0.003261,0.000000,0.658696,0.661957,0.063239,99.000000,1
2,192.168.100.5,2025-10-20 14:35:00,assets-msn-com-world-atm-default.trafficmanage...,36.0,5c290cb2d304959c6b941e8f728a610d.azr.footprint...,4.562365,3.966666,4.562365,51,53,...,0.125133,375.6,6.260000,0.119276,0.000000,0.652822,0.772098,0.124847,99.000000,1
3,192.168.100.5,2025-09-20 13:20:00,ccs.newrocktech.com,48.0,27fae813bd3f98110d447d86f7a785b0c4269e23.malwa...,4.542302,3.321104,4.542302,19,63,...,0.117342,424.4,7.073333,0.265787,0.000000,0.659755,0.925542,0.102258,99.000000,1
4,192.168.100.5,2025-10-05 14:05:00,21d40f967eb5ee5a9f4079b3893e7f7d.azr.footprint...,12.0,21d40f967eb5ee5a9f4079b3893e7f7d.azr.footprint...,4.515380,4.515380,4.515380,53,53,...,0.134146,65.6,1.093333,0.018293,0.000000,0.658537,0.676829,0.074190,99.000000,1
5,192.168.100.5,2025-10-03 08:05:00,722b5234244f0cfd2dfe834d317f1b71b9b9a142.malwa...,12.0,5c1f5499453db34076d7382fb6555e8a8e69ab70.malwa...,4.506355,4.407536,4.506355,63,63,...,0.125806,124.0,2.066667,0.225806,0.225806,0.509677,0.961290,0.125595,99.000000,1
6,192.168.100.5,2025-10-03 07:55:00,safebrowsing.googleapis.com,12.0,5c1f5499453db34076d7382fb6555e8a8e69ab70.malwa...,4.506355,3.810081,4.506355,27,63,...,0.157035,159.2,2.653333,0.324121,0.000000,0.635678,0.959799,0.090968,99.000000,1
7,192.168.100.5,2025-10-03 12:15:00,a1dc74ef8495c9a1489dd937659b5c2875027e16.malwa...,8.0,a1dc74ef8495c9a1489dd937659b5c2875027e16.malwa...,4.463657,4.463657,4.463657,63,63,...,0.157895,7.6,0.126667,0.052632,0.000000,0.631579,0.684211,0.073007,99.000000,1
8,192.168.100.5,2025-09-25 12:20:00,part-0044.t-0009.fb-t-msedge.net,44.0,8d69bf802e91ebb07b84e6253c2cf962.cdn.edge.soto...,4.297152,3.777518,4.297152,32,51,...,0.085475,358.0,5.966667,0.005587,0.003352,0.640782,0.649721,0.096670,99.000000,1
9,10.10.10.40,2025-10-11 15:00:00,a.c.d.2.c.6.e.f.f.f.b.f.7.7.8.c.0.0.0.0.0.0.0....,21.0,www.google.com,2.842371,2.757562,2.842371,72,14,...,0.060606,6.6,0.110000,0.000000,0.000000,0.000000,0.000000,0.064801,99.000000,1


Source IP,Window Start,TopDomain,TopDomainCount,HiEntropyDomain,HiEntropyValue,top1_entropy,hiE_entropy,top1_length,hiE_length,top1_digit_ratio,hiE_digit_ratio,top1_vowel_ratio,hiE_vowel_ratio,top1_share,avg_entropy,unique_domain_ratio,qpm,qps,nxdomain_ratio,servfail_ratio,refused_ratio,error_ratio,if_score_combined,if_severity,if_label
10.10.10.69,2025-09-20 13:05:00,content-autofill.googleapis.com,30.0,xga4p6accpq7k2gopuvq-f-6cb4e5303-clientnsv4-s.akamaihd.net,4.709202,3.821312,4.709202,31,58,0.000000,0.189655,0.387097,0.206897,0.063158,3.468078,0.183158,95.0,1.583333,0.000000,0.008421,0.000000,0.008421,0.012037,99.000000,1
192.168.100.5,2025-10-07 11:45:00,a1968.i6g1.akamai.net,24.0,activationedge-fabrick-1124129460.us-west-2.elb.amazonaws.com,4.573459,3.499228,4.573459,21,61,0.285714,0.180328,0.333333,0.278689,0.013043,3.554274,0.092391,368.0,6.133333,0.003261,0.000000,0.658696,0.661957,0.063239,99.000000,1
192.168.100.5,2025-10-20 14:35:00,assets-msn-com-world-atm-default.trafficmanager.net,36.0,5c290cb2d304959c6b941e8f728a610d.azr.footprintdns.com,4.562365,3.966666,4.562365,51,53,0.000000,0.415094,0.274510,0.132075,0.019169,3.921472,0.125133,375.6,6.260000,0.119276,0.000000,0.652822,0.772098,0.124847,99.000000,1
192.168.100.5,2025-09-20 13:20:00,ccs.newrocktech.com,48.0,27fae813bd3f98110d447d86f7a785b0c4269e23.malware.hash.cymru.com,4.542302,3.321104,4.542302,19,63,0.000000,0.428571,0.210526,0.158730,0.022620,4.174529,0.117342,424.4,7.073333,0.265787,0.000000,0.659755,0.925542,0.102258,99.000000,1
192.168.100.5,2025-10-05 14:05:00,21d40f967eb5ee5a9f4079b3893e7f7d.azr.footprintdns.com,12.0,21d40f967eb5ee5a9f4079b3893e7f7d.azr.footprintdns.com,4.515380,4.515380,4.515380,53,53,0.377358,0.377358,0.188679,0.188679,0.036585,3.621282,0.134146,65.6,1.093333,0.018293,0.000000,0.658537,0.676829,0.074190,99.000000,1
192.168.100.5,2025-10-03 08:05:00,722b5234244f0cfd2dfe834d317f1b71b9b9a142.malware.hash.cymru.com,12.0,5c1f5499453db34076d7382fb6555e8a8e69ab70.malware.hash.cymru.com,4.506355,4.407536,4.506355,63,63,0.412698,0.444444,0.126984,0.158730,0.019355,4.287940,0.125806,124.0,2.066667,0.225806,0.225806,0.509677,0.961290,0.125595,99.000000,1
192.168.100.5,2025-10-03 07:55:00,safebrowsing.googleapis.com,12.0,5c1f5499453db34076d7382fb6555e8a8e69ab70.malware.hash.cymru.com,4.506355,3.810081,4.506355,27,63,0.000000,0.444444,0.370370,0.158730,0.015075,4.309988,0.157035,159.2,2.653333,0.324121,0.000000,0.635678,0.959799,0.090968,99.000000,1
192.168.100.5,2025-10-03 12:15:00,a1dc74ef8495c9a1489dd937659b5c2875027e16.malware.hash.cymru.com,8.0,a1dc74ef8495c9a1489dd937659b5c2875027e16.malware.hash.cymru.com,4.463657,4.463657,4.463657,63,63,0.444444,0.444444,0.158730,0.158730,0.210526,3.680858,0.157895,7.6,0.126667,0.052632,0.000000,0.631579,0.684211,0.073007,99.000000,1
192.168.100.5,2025-09-25 12:20:00,part-0044.t-0009.fb-t-msedge.net,44.0,8d69bf802e91ebb07b84e6253c2cf962.cdn.edge.sotoon.ir,4.297152,3.777518,4.297152,32,51,0.250000,0.392157,0.125000,0.176471,0.024581,3.469921,0.085475,358.0,5.966667,0.005587,0.003352,0.640782,0.649721,0.096670,99.000000,1
10.10.10.40,2025-10-11 15:00:00,a.c.d.2.c.6.e.f.f.f.b.f.7.7.8.c.0.0.0.0.0.0.0.0.0.0.0.0.0.8.e.f.ip6.arpa,21.0,www.google.com,2.842371,2.757562,2.842371,72,14,0.277778,0.000000,0.083333,0.285714,0.636364,2.788402,0.060606,6.6,0.110000,0.000000,0.000000,0.000000,0.000000,0.064801,99.000000,1
